
<br>
<font>
<div dir=ltr align=center>
<font color=0F5298 size=7>
    System 2 - Homework 2<br>
<font color=2565AE size=5>
    Spring 2025<br>
<font color=3C99D size=5>
    Inference-time Scaling <br>


Student Number = 403210725

Name = Farzan Rahmani

# Assignment Overview (Part 1)

In this assignment, you will explore inference scaling techniques in large language models (LLMs) and evaluate their performance using the Math Benchmark. Throughout the notebook, you will learn about several inference methods, including:

- **Chain-of-Thought (CoT):** A method where the model generates intermediate reasoning steps before providing the final answer.
- **Best-of-n Sampling:** An approach that generates multiple candidate responses and selects the best one based on a scoring function.
- **Beam Search:** A technique that expands several possible sequences simultaneously, choosing the most promising ones based on probability.
- **Self-Refinement:** An iterative process where the model revises its output to improve accuracy and coherence.

The **Math Benchmark** is a suite of challenging mathematical problems designed to test the reasoning and problem-solving capabilities of LLMs. The benchmark includes a variety of questions ranging from basic arithmetic and algebra to more advanced topics such as geometry and calculus. For example, you might be asked to solve an equation like `2x + 5 = 15` or compute the derivative of a function, tasks that assess the model's ability to handle both straightforward and complex mathematical queries.

By the end of this assignment, you will have:
- Gained a deeper understanding of inference time scaling methods in LLMs.
- Compared the effectiveness of different inference techniques using a rigorous math evaluation framework.

Let's dive into the notebook and begin exploring how these methods perform on a challenging set of math problems!


# installing Dependencies

In [1]:
!pip install vllm
!pip install transformers accelerate datasets

!pip install -U "datasets>=2.19.1" "huggingface_hub>=0.21.2" "fsspec>=2024.1"

from IPython.display import clear_output
clear_output()

* You should use this cell if you're running the notebook on Google Colab. If you're using Kaggle, you don't need to run this cell.

In [ ]:
!pip install --upgrade numpy -q
import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 72.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cuml-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
cudf-cu12 25.2.1 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
dask-cuda 25.2.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.
ydf 0.11.0 requires protobuf<6.0.0,>=5.29.1, but you have protobuf 4.25.7 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
distributed-ucxx-cu12 0.42.0 requires numba<0.61.0a0,>=0.59.1, but you have numba 0.61.2 which is incompatible.


## vLLM: Accelerated Inference Engine for LLMs

vLLM is an open-source project designed to optimize the loading and inference of large language models. By leveraging advanced memory management techniques and dynamic batching, vLLM significantly speeds up the inference process, making it easier to deploy and experiment with LLMs even on hardware with limited resources.
So we use vLLM to get results faster.

## VLLM Server Setup and Initialization

In this section, we install the required packages, ensure that only one server instance is running, and start the VLLM server using the model `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B`.

**Installation and Cleanup:**
- The necessary packages (`vllm`, `transformers`, `accelerate`, and `datasets`) are installed in a cell with hidden output to keep the notebook clean.
- Any previously running VLLM server instances are terminated before starting a new one. This prevents multiple servers from running simultaneously.

**Server Initialization:**
- The server is launched as a background process using `subprocess.Popen`.
- **Initialization Time:**  
  The server typically takes about **1 minute** to fully initialize.
- **GPU Memory Utilization:**  
  Monitor your GPU memory usage. Initially, it will be at **0 GB**, and then it will gradually increase until it reaches approximately **12 GB** when the server is fully up and running.

Please wait until the GPU memory stabilizes around **12 GB** before proceeding to the next steps.


In [1]:
import subprocess

# Kill any running VLLM server instances for the specified model
kill_cmd = "pkill -f 'vllm serve deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'"
subprocess.run(kill_cmd, shell=True)

# Command to start the VLLM server
cmd = [
    "vllm", "serve", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "--port", "8000", "--dtype=half", "--max-model-len", "5192"
]
server_process = subprocess.Popen(cmd)

print("Server started!")

Server started!


* you can debug last cell if doesn't work right with this cell (if that works you DO NOT run this cell)

In [ ]:
!vllm serve "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"   --port 8000   --dtype=half   --max-model-len 5192

# Helper Functions Overview

This section contains a series of helper functions designed to facilitate the evaluation of mathematical problem solving using the MATH-500 dataset and a local LLM server. These functions handle tasks such as dataset loading, answer extraction, normalization of various mathematical expressions, answer comparison, and result management. Below is an explanation of each group of functions:

---

## Dataset Loading

- **`load_math500_dataset()`**  
  Loads the test split of the MATH-500 dataset from the Hugging Face repository (`HuggingFaceH4/MATH-500`). This dataset provides the math problems and corresponding solutions used for evaluation.

---

## Answer Extraction

- **`extract_answer(response: str) -> Optional[str]`**  
  Searches the provided text for the last occurrence of the LaTeX command `\boxed{...}` and extracts the content within it. This function is essential for retrieving the final answer from the formatted solutions.

---

## Normalization Functions

Normalization functions standardize the format of mathematical expressions to enable accurate comparisons between predicted and correct answers. These functions account for various representations, ensuring that equivalent answers written in different formats are recognized as equal.

- **`normalize_number(num_str: str) -> str`**  
  Cleans and normalizes numeric strings by removing extraneous characters (e.g., commas, currency symbols, and measurement units) and formatting them into a consistent number format.

- **`numerically_equal(str1: str, str2: str) -> bool`**  
  Checks if two numeric strings represent the same value within a small tolerance, accounting for floating point precision issues.

- **`normalize_fraction(fraction_str: str) -> str`**  
  Converts various representations of fractions (with or without braces or using a slash) into a standard LaTeX format: `\frac{numerator}{denominator}`.

- **`normalize_matrix_entry(entry: str) -> str`**  
  Standardizes individual matrix entries, especially handling fractions and slash-separated numbers, to ensure consistency within matrix representations.

- **`normalize_matrix(matrix_str: str) -> str`**  
  Processes a LaTeX matrix (formatted with `\begin{pmatrix}` and `\end{pmatrix}`) by normalizing each row and each entry using the matrix entry normalization.

- **`normalize_algebraic_expression(expr: str) -> str`**  
  Standardizes algebraic expressions by handling coefficients, variables, exponents, and special terms like π (pi). This helps compare algebraic answers regardless of minor formatting differences.

- **`normalize_interval_bound(bound: str) -> str`**  
  Normalizes the boundary of an interval, ensuring that symbols like infinity (`\infty`) and other numeric boundaries are consistently formatted.

- **`normalize_interval(interval_str: str) -> str`**  
  Standardizes an interval provided in LaTeX, ensuring that both bounds are normalized and that the overall format (including brackets) is consistent.

- **`normalize_ordered_tuple(tuple_str: str) -> str`**  
  Normalizes an ordered tuple by splitting its elements and applying answer normalization to each component, ensuring a standard tuple representation.

- **`normalize_answer(answer: str) -> str`**  
  The central normalization function that applies the various normalization steps to a given answer. It cleans up LaTeX formatting, removes unnecessary spaces, and calls the specialized normalization functions to standardize numeric, fractional, algebraic, and other mathematical expressions.

---

## Answer Comparison

- **`compare_answers(correct_answer: str, predicted_answer: Optional[str]) -> bool`**  
  Compares the normalized versions of the correct answer and the predicted answer. This function ensures that answers are compared in a standardized format so that minor differences in formatting do not affect the evaluation outcome.

---

## Result Management Functions

These functions handle saving and analyzing the results of the evaluation process.

- **`load_existing_results(filename: str) -> list[Dict]`**  
  Loads previously saved evaluation results from a JSON file. If the file does not exist, it returns an empty list.

- **`save_result(filename: str, result: Dict)`**  
  Appends a single evaluation result (including problem details, the LLM response, and correctness) to the results file in JSON format.

- **`analyze_results(results: list[Dict])`**  
  Analyzes the evaluation outcomes by summarizing the total number of problems, counting the correct answers, calculating the accuracy, and printing details for any problems that were answered incorrectly.

---

## Main Evaluation and Response Handling

- **`evaluate()`**  
  The primary function that orchestrates the evaluation process:
  - Creates a results directory if it doesn't already exist.
  - Loads the MATH-500 dataset.
  - Iterates over each problem (while skipping already processed ones).
  - Sends the problem text to the local LLM server using `get_llm_response`.
  - Extracts and compares the answers, then saves the result.
  - Finally, it analyzes and prints a summary of the evaluation.

- **`get_llm_response(prompt: str) -> str`**  
  Sends a prompt to the locally running LLM server (via an HTTP POST request to `http://localhost:8000/v1/chat/completions`) and returns the server's response. This function is key to obtaining the model's predicted answer.


In [2]:
import json
import os
import re
from typing import Dict, Optional, Union
from datasets import load_dataset
from tqdm import tqdm
import torch

# Load the MATH-500 dataset
def load_math500_dataset():
    dataset = load_dataset("HuggingFaceH4/MATH-500")["test"]
    return dataset

# Extract the last boxed answer from text
def extract_answer(response: str) -> Optional[str]:
    if not response:
        return None
    start_idx = response.rfind('\\boxed{')
    if start_idx == -1:
        return None
    brace_count = 1
    pos = start_idx + 7  # length of '\boxed{'
    while pos < len(response) and brace_count > 0:
        if response[pos] == '{':
            brace_count += 1
        elif response[pos] == '}':
            brace_count -= 1
        pos += 1
    if brace_count == 0:
        answer = response[start_idx + 7:pos - 1]
        return answer.strip()
    return None

# Normalization and comparison functions (unchanged from original)
def normalize_number(num_str: str) -> str:
    try:
        cleaned = re.sub(r'[,\$\\]|\s*(?:cm|m|kg|ft|in|lb|oz|ml|L)$|\s*\\text{[^}]+}', '', num_str).strip()
        if cleaned.startswith('.'):
            cleaned = '0' + cleaned
        num = float(cleaned)
        if abs(num) < 1 and '.' in cleaned:
            decimal_places = len(cleaned.split('.')[1])
            format_str = f"{{:.{decimal_places}f}}"
            result = format_str.format(num)
        else:
            result = str(num)
        return result
    except:
        return num_str

def numerically_equal(str1: str, str2: str) -> bool:
    try:
        return abs(float(str1) - float(str2)) < 1e-10
    except:
        return False

def normalize_fraction(fraction_str: str) -> str:
    try:
        fraction_str = fraction_str.replace('\\dfrac', '\\frac')
        fraction_str = ''.join(fraction_str.split())
        fraction_str = re.sub(r'\s*\\text{[^}]+}', '', fraction_str)
        mixed_brace = re.match(r'^\\frac(\d+)\{(\d+)\}$', fraction_str)
        if mixed_brace:
            num, den = mixed_brace.groups()
            return f"\\frac{{{num}}}{{{den}}}"
        no_braces = re.match(r'^\\frac(\d+)(\d+)$', fraction_str)
        if no_braces:
            num, den = no_braces.groups()
            return f"\\frac{{{num}}}{{{den}}}"
        if '/' in fraction_str and not any(c in fraction_str for c in '\\{}'):
            num, den = fraction_str.split('/')
            return f"\\frac{{{num.strip()}}}{{{den.strip()}}}"
        standard = re.match(r'^\\frac\{([^{}]+)\}\{([^{}]+)\}$', fraction_str)
        if standard:
            num, den = standard.groups()
            return f"\\frac{{{num}}}{{{den}}}"
    except:
        return fraction_str

def normalize_matrix_entry(entry: str) -> str:
    entry = ''.join(entry.split())
    if '/' in entry and not any(c in entry for c in '\\{}'):
        if entry.startswith('-'):
            num, den = entry[1:].split('/')
            return f"-{num.strip()}/{den.strip()}"
        else:
            num, den = entry.split('/')
            return f"{num.strip()}/{den.strip()}"
    entry = entry.replace('\\dfrac', '\\frac')
    frac_match = re.match(r'^(-)?\\frac\{(\d+)\}\{(\d+)\}$', entry)
    if frac_match:
        sign, num, den = frac_match.groups()
        sign = sign if sign else ''
        return f"{sign}{num}/{den}"
    return entry

def normalize_matrix(matrix_str: str) -> str:
    try:
        matrix_str = ''.join(matrix_str.split())
        match = re.match(r'^\\begin\{pmatrix\}(.*?)\\end\{pmatrix\}$', matrix_str)
        if not match:
            return matrix_str
        content = match.group(1)
        rows = content.split('\\\\')
        normalized_rows = []
        for row in rows:
            if '&' in row:
                entries = [normalize_matrix_entry(entry) for entry in row.split('&')]
            else:
                entries = [normalize_matrix_entry(row)]
            normalized_rows.append('&'.join(entries))
        result = "\\begin{pmatrix}" + "\\\\".join(normalized_rows) + "\\end{pmatrix}"
        return result
    except:
        return matrix_str

def normalize_algebraic_expression(expr: str) -> str:
    try:
        expr = ''.join(expr.split())
        monomial_match = re.match(r'^(-?\d*\.?\d*)?([a-zA-Z])(?:\^(-?\d+))?$', expr)
        if monomial_match:
            coeff, var, exp = monomial_match.groups()
            coeff = coeff if coeff and coeff not in ['+', '-'] else ('1' if not coeff else '-1')
            exp = exp if exp else '1'
            if coeff == '1' and exp == '1':
                return var
            elif coeff == '1':
                return f"{var}^{exp}"
            elif coeff == '-1' and exp == '1':
                return f"-{var}"
            elif coeff == '-1':
                return f"-{var}^{exp}"
            elif exp == '1':
                return f"{coeff}{var}"
            else:
                return f"{coeff}{var}^{exp}"
        pi_term_match = re.match(r'^(-?\d*\.?\d*)\\?pi$', expr)
        if pi_term_match:
            coeff = pi_term_match.group(1)
            if not coeff or coeff == '-':
                coeff = '-1' if coeff == '-' else '1'
            return f"{coeff}\\pi"
        frac_pi_match = re.match(r'^\\frac{([^{}]+)}{([^{}]+)}\\?pi$', expr)
        if frac_pi_match:
            num, den = frac_pi_match.groups()
            return f"\\frac{{{num}}}{{{den}}}\\pi"
        frac_match = re.match(r'^\\frac{([^{}]+)}{([^{}]+)}$', expr)
        if frac_match:
            num, den = frac_match.groups()
            return f"\\frac{{{num}}}{{{den}}}"
    except:
        return expr.lower()

def normalize_interval_bound(bound: str) -> str:
    if '\\infty' in bound:
        sign = '-' if bound.startswith('-') else ''
        return f"{sign}\\infty"
    return normalize_answer(bound) or bound

def normalize_interval(interval_str: str) -> str:
    try:
        interval_str = ''.join(interval_str.split())
        match = re.match(r'^\\left?([\[\(])(.*?),(.*?)\\right?([\]\)])$', interval_str)
        if not match:
            match = re.match(r'^([\[\(])(.*?),(.*?)([\]\)])$', interval_str)
            if not match:
                return interval_str
        left_bracket, left_bound, right_bound, right_bracket = match.groups()
        norm_left = normalize_interval_bound(left_bound)
        norm_right = normalize_interval_bound(right_bound)
        return f"\\left{left_bracket}{norm_left},{norm_right}\\right{right_bracket}"
    except:
        return interval_str

def normalize_ordered_tuple(tuple_str: str) -> str:
    try:
        tuple_str = tuple_str.replace('\\dfrac', '\\frac')
        tuple_str = tuple_str.replace('\\left', '').replace('\\right', '')
        tuple_str = re.sub(r'\\?\s+', '', tuple_str)
        inner = tuple_str.strip('()')
        parts = inner.split(',')
        normalized_parts = [normalize_answer(part.strip()) for part in parts if normalize_answer(part.strip())]
        return f"({','.join(normalized_parts)})"
    except:
        return None

def normalize_answer(answer: str) -> str:
    if answer is None:
        return ""
    answer = re.sub(r'\\text{[^}]+(?:inches|feet|meters|cm|m|kg|ft|in|lb|oz|ml|L|per|second|minute|hour)[^}]*}', '', answer)
    answer = re.sub(r'(?<!\\)\s+', '', answer)
    ordered_pair_match = re.match(r'^(?:\\left)?\((.*?)(?:\\right)?\)$', answer)
    if ordered_pair_match:
        content = ordered_pair_match.group(1)
        parts = content.split(',')
        normalized_parts = [normalize_answer(part) for part in parts if normalize_answer(part)]
        return f"({','.join(normalized_parts)})"
    answer = ''.join(answer.split())
    if not answer:
        return None
    pm_match = re.match(r'^(.*?)(?:\\pm|-)(.*?)$', answer)
    if pm_match:
        left, right = pm_match.groups()
        norm_left = normalize_answer(left) if left else ""
        norm_right = normalize_answer(right) if right else ""
        if norm_left or norm_right:
            return f"{norm_left}\\pm{norm_right}"
    trig_match = re.match(r'^\\(?:sin|cos|tan|cot|sec|csc)\s*([a-zA-Z])$', answer)
    if trig_match:
        variable = trig_match.group(1)
        func_name = re.match(r'^\\(.*?)(?:\s|$)', answer).group(1)
        return f"\\{func_name}{variable}"
    text_match = re.match(r'^(?:\\text{)?([A-Za-z]+)(?:})?$', answer)
    if text_match:
        return text_match.group(1).lower()
    if (answer.startswith('\\left[') or answer.startswith('\\left(') or
        answer.startswith('[') or answer.startswith('(')) and \
       (answer.endswith('\\right]') or answer.endswith('\\right)') or
        answer.endswith(']') or answer.endswith(')')):
        return normalize_interval(answer)
    if answer.startswith('\\begin{pmatrix}') and answer.endswith('\\end{pmatrix}'):
        return normalize_matrix(answer)
    answer = answer.replace('\\dfrac', '\\frac')
    if '\\frac' in answer or '/' in answer:
        return normalize_fraction(answer)
    neg_sqrt_match = re.match(r'^-\\sqrt\{?(\d+)\}?$', answer)
    if neg_sqrt_match:
        num = neg_sqrt_match.group(1)
        return f"-\\sqrt{{{num}}}"
    sqrt_match = re.match(r'^(\d*)?\\sqrt\{?(\d+)\}?$', answer)
    if sqrt_match:
        coeff, num = sqrt_match.groups()
        coeff = coeff if coeff else '1'
        return f"\\sqrt{{{num}}}" if coeff == '1' else f"{coeff}\\sqrt{{{num}}}"
    sqrt_with_coeff_match = re.match(r'^(\d+)\\sqrt\{?(\d+)\}?$', answer)
    if sqrt_with_coeff_match:
        coeff, num = sqrt_with_coeff_match.groups()
        return f"{coeff}\\sqrt{{{num}}}"
    base_match = re.match(r'^(\d+)(?:_\{?(\d+)\}?|_(\d+))$', answer)
    if base_match:
        number, base1, base2 = base_match.groups()
        base = base1 if base1 else base2
        return f"{number}_{base}"
    percent_match = re.match(r'^(\d+(?:\.\d*)?)\s*\\?%$', answer)
    if percent_match:
        return normalize_number(percent_match.group(1))
    unit_match = re.match(r'^(\d+(?:\.\d*)?)\s*(?:(?:\\[,\s])|,)?\s*(?:\\\\)?(?:\\text{(\w+)}|\\?(?:cm|m|kg|ft|in|lb|oz|ml|L))$', answer)
    if unit_match:
        return normalize_number(unit_match.group(1))
    currency_match = re.match(r'^\\?\$?([\d,]+\.?\d*)$', answer)
    if currency_match:
        return normalize_number(currency_match.group(1))
    if re.match(r'^-?[\d,]+$', answer):
        return normalize_number(answer)
    unit_match = re.match(r'^(-?[\d,]+(?:\.\d*)?)\s*(?:\\(?:mbox|text|hbox|displaystyle)\{[^}]+\})?(?:\^?\d)?$', answer)
    if unit_match:
        return normalize_number(unit_match.group(1))
    mc_match = re.match(r'^\\text{\(?([A-Za-z])\)?}$|^\(?([A-Za-z])\)?$', answer)
    if mc_match:
        return (mc_match.group(1) or mc_match.group(2)).lower()
    degree_match = re.match(r'^(-?[\d,]+(?:\.\d*)?)\s*(?:(?:\^?\\circ)|(?:{\\circ})|(?:°))?$', answer)
    if degree_match:
        return normalize_number(degree_match.group(1))
    answer = re.sub(r'\\text{([^{}]+)}', r'\1', answer)
    try:
        return normalize_algebraic_expression(answer)
    except:
        pass
    answer = answer.replace('\\left', '').replace('\\right', '')
    answer = answer.replace('\\(', '(').replace('\\)', ')')
    answer = answer.replace('\\[', '[').replace('\\]', ']')
    answer = answer.replace('\\{', '{').replace('\\}', '}')
    answer = re.sub(r'\\sqrt\{?(\d+)\}?', r'\\sqrt{\1}', answer)
    answer = re.sub(r'\\sqrt{([^{}]+)}', r'\\sqrt\1', answer)
    if re.match(r'^\d+\\%$', answer) or re.match(r'^\d+$', answer):
        answer = re.sub(r'\\%$', '', answer)
    answer = re.sub(r'\\text{([^{}]+)}', r'\1', answer)
    while len(answer) >= 2 and answer[0] == '{' and answer[-1] == '}':
        if '\\frac' in answer:
            break
        answer = answer[1:-1]
    return answer.lower() if answer else None

def compare_answers(correct_answer: str, predicted_answer: Optional[str]) -> bool:
    if predicted_answer is None:
        return False
    if numerically_equal(correct_answer, predicted_answer):
        return True
    normalized_correct = normalize_answer(correct_answer)
    normalized_predicted = normalize_answer(predicted_answer)
    if not normalized_correct or not normalized_predicted:
        return False
    if normalized_correct == "" and normalized_predicted == "":
        return False
    if ('\\left[' in normalized_correct or '\\left(' in normalized_correct) and \
       ('\\left[' in normalized_predicted or '\\left(' in normalized_predicted):
        return normalized_correct == normalized_predicted
    return normalized_correct == normalized_predicted

# Load existing results
def load_existing_results(filename: str) -> list[Dict]:
    try:
        with open(filename, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return []

# Save a single result
def save_result(filename: str, result: Dict):
    results = load_existing_results(filename)
    results.append(result)
    with open(filename, 'w') as f:
        json.dump(results, f, indent=2)

# Analyze and print results
def analyze_results(results: list[Dict]):
    total = len(results)
    correct = sum(1 for r in results if r['is_correct'])
    accuracy = correct / total if total > 0 else 0
    print("\n=== Results Summary ===")
    print(f"Total problems: {total}")
    print(f"Correct answers: {correct}")
    print(f"Accuracy: {accuracy:.2%}")
    print("\n=== Incorrect Problems ===")
    for r in results:
        if not r['is_correct']:
            print(f"Problem {r['index']}:")
            print(f"Expected: {r['correct_answer']}")
            print(f"Predicted: {r['predicted_answer']}")
            print("---")

# Main evaluation function
def evaluate():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    t=0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes:
            continue
        t += 1
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])  # Extract from 'solution', not 'answer'
        response = get_llm_response(problem_text)
        predicted_answer = extract_answer(response)
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"cnt :  {cnt} idx: {t}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)


## LLM Query Function

* This Python function sends prompts to a locally-hosted LLM API and returns the generated response
* you can change max_tokens and temperature as you want

In [3]:
import requests

def get_llm_response(prompt):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    # "max_tokens": 500,
    "max_tokens": 2000,
    "temperature": 0.6
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

## Test Prompt: Evaluating an Integral

In this cell, we define a new math benchmark question to verify that the LLM server is correctly set up and that responses can be retrieved.

**Question:**  
What is the value of the integral  
$$\int_0^1 x^2\,dx$$  

**Expected Answer:**  
$$\boxed{\frac{1}{3}}$$

The cell sends this prompt to the LLM server using the `get_llm_response` function and prints the response. This helps confirm that the integration between the notebook and the LLM server is working properly.

In [ ]:
# Define a new math benchmark question for testing
question = "What is the value of the integral $$\\int_0^1 x^2 dx$$ answer it directly in one sentence?"
# Real answer: \boxed{\frac{1}{3}}

# Get response from the LLM server using the provided get_llm_response function
response = get_llm_response(question)

# Print the response to verify that the setup is working correctly
print("Response:", response)


Response: To evaluate the integral of \( x^2 \) from 0 to 1, I recognize that this is a basic power rule integral.

First, I apply the power rule for integration, which states that the integral of \( x^n \) is \( \frac{x^{n+1}}{n+1} \) plus a constant.

Applying this to \( x^2 \), the integral becomes \( \frac{x^3}{3} \).

Next, I evaluate this antiderivative at the upper limit of 1 and the lower limit of 0.

Calculating at 1: \( \frac{1^3}{3} = \frac{1}{3} \).

Calculating at 0: \( \frac{0^3}{3} = 0 \).

Finally, subtracting the lower limit result from the upper limit result gives \( \frac{1}{3} - 0 = \frac{1}{3} \).

Therefore, the value of the integral \( \int_0^1 x^2 dx \) is \( \frac{1}{3} \).
</think>

The value of the integral is:

\[
\boxed{\dfrac{1}{3}}
\]


# Customizable CoT Prompt Template
* modify CoT prompt then evaluate on math benchmark


In [ ]:
import requests

# Define the system prompt
COT_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''

def get_COT_response(problem):
    prompt = COT_PROMPT + "\n" + problem
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    "max_tokens": 2000,
    # "temperature": 0.3
    "temperature": 0.1
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

# Evaluate CoT
* modify response generation part to evalute this method.

In [ ]:
def evaluate_cot():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_cot.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes:
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        ##########################################################
        response = get_COT_response(problem_text)
        predicted_answer = extract_answer(response)
        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [ ]:
evaluate_cot() # the given accuracy: Accuracy: 40.00% (12/30) # change hyperparams or prompts to get better results

Evaluating problems:   0%|          | 1/500 [00:07<1:00:40,  7.30s/it]

corrects :  0 idx: 0


Evaluating problems:   0%|          | 2/500 [00:39<2:59:47, 21.66s/it]

corrects :  0 idx: 1


Evaluating problems:   1%|          | 3/500 [00:47<2:09:28, 15.63s/it]

corrects :  1 idx: 2


... (50 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [10:22<2:58:02, 22.68s/it]

corrects :  13 idx: 28


Evaluating problems:   6%|▌         | 30/500 [10:30<2:44:30, 21.00s/it]

corrects :  14 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 14
Accuracy: 46.67%

=== Incorrect Problems ===
Problem 0:
Expected: \left( 3, \frac{\pi}{2} \right)
Predicted: \left(3,\ \dfrac{\pi}{2}\right)
---
Problem 1:
Expected: p - q
Predicted: None
---
Problem 9:
Expected: 4
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: 280
---
Problem 15:
Expected: 6 - 5i
Predicted: None
---
Problem 17:
Expected: \pi
Predicted: None
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 19:
Expected: 3
Predicted: None
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: None
---
Problem 22:
Expected: 5
Predicted: None
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 25:
Expected: 1,-2
Predicted: None
---
Problem 26:
Expected: 144
Predicted: None
---
Problem 28:
Expected: -2 + 7i
Predicted: -2 + 7i
---


## Best-of-N

The **Best-of-N** approach improves math problem-solving by generating *N* solutions and selecting the one with the highest average token log-likelihood. Each solution is crafted using a prompt that encourages step-by-step reasoning and includes a formatted answer. The final selected response is both reliable and well-presented.

### Steps
1. **Generate**: Produce *N* responses using a structured guiding prompt.
2. **Evaluate**: Compute the average log-likelihood for each response based on token probabilities.
3. **Select**: Identify and choose the response with the highest score.

This method ensures a statistically robust and clearly formatted solution.


## Verification Methods in Best‑of‑N Evaluation

When sampling multiple candidate solutions for each math problem, we need a reliable way to choose the single best answer. We support two complementary approaches:

### Log‑Probability Scoring

**Concept**  
Each generated solution comes with token‑level log‑likelihoods. By averaging these values across all tokens in the response, we obtain a single score reflecting how “confident” the model is in that entire output.

**Why Use It**  
- **Self‑Contained & Fast**: Requires no external calls or additional models.  
- **Cost‑Effective**: Purely internal computation, so it adds negligible expense.  

**Limitations**  
- A high likelihood does not always imply a correct or well‑reasoned solution, especially on complex math problems.

---

### LLM‑Based Verification

**Concept**  
Instead of trusting raw likelihoods, we hand all sampled responses off to a second, high‑quality language model (e.g. Gemini Mini). That model reads the original problem and the list of candidate boxed answers, then selects the one it judges to be correct.

**Why Use It**  
- **Deeper Reasoning**: A dedicated verifier can compare alternative answers and catch subtle mistakes.  
- **Improved Robustness**: Mitigates cases where a flawed but high‑probability output would otherwise be chosen.

**Trade‑Offs**  
- **Slower**: Requires additional API calls and round‑trip latency.  
- **External Cost**: Incurs usage fees on the verification model.

---

### Balancing Speed, Cost, and Accuracy

By exposing a simple toggle between these two methods, you can:

- **Optimize for Speed**: Use log‑prob scoring when you need rapid, low‑cost evaluation.  
- **Optimize for Accuracy**: Use LLM‑based verification when correctness is paramount.  

Experiment on your dataset to find the right trade‑off for your needs.

In [4]:
from openai import OpenAI
from google.colab import userdata

# ⚠️ Replace this with your real Gemini API key
client = OpenAI(
    api_key=userdata.get('Gemeni_API_KEY'),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
# Use the cheapest Gemini model; swap to another if you like
# LLM_API_MODEL = "gemini-mini"
LLM_API_MODEL = "gemini-2.0-flash"
# LLM_API_MODEL = "gemini-2.0-flash-lite"
# LLM_API_MODEL = "gemini-2.5-flash-preview-04-17"
# LLM_API_MODEL = "gemini-2.5-pro-preview-03-25"

def get_api_response(prompt: str) -> str:
    """
    Send `prompt` to Gemini Mini and return its reply.
    """
    resp = client.chat.completions.create(
        model=LLM_API_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=1024,
    )
    return resp.choices[0].message.content

# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# import openai
# from google.colab import userdata
# import google.generativeai as genai

# # ⚠️ Replace this with your real Gemini API key
# openai.api_key = userdata.get('Gemeni_API_KEY')
# genai.configure(api_key=userdata.get('Gemeni_API_KEY'))

# # Use the cheapest Gemini model; swap to another if you like
# # LLM_API_MODEL = "gemini-mini"
# # LLM_API_MODEL = "gemini-2.0-flash"
# # LLM_API_MODEL = "gemini-2.0-flash-lite"
# LLM_API_MODEL = "gemini-2.5-flash-preview-04-17"
# # LLM_API_MODEL = "gemini-2.5-pro-preview-03-25"

# # Initialize the model
# API_model = genai.GenerativeModel(
#     model_name="gemini-2.5-flash-preview-04-17",
#     generation_config={
#         "temperature": 0.0,
#         "max_output_tokens": 1024,
#     }
# )

# def get_api_response(prompt: str) -> str:
#     """
#     Send `prompt` to Gemini Mini and return its reply.
#     """
#     response = API_model.generate_content(prompt)
#     return response.text
# # ───────────────────────────────────────────────────────────────────────────────

In [ ]:
import re
import time
import math
import heapq
import requests
from typing import List, Optional


SYSTEM_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''

def verify_with_gpt(problem: str, outputs: List[str]) -> Optional[str]:
    """
    Given the original problem and a list of candidate full-response texts,
    asks a GPT API to pick the correct final answer (boxed).
    """
    # TODO: Deduplicate `outputs` into a `unique_answers: List[str]` by extracting
    #       each boxed answer via `extract_answer` (use "<no_boxed_answer>" if none).
    # TODO: Build `options` as numbered lines of the form "1. \\boxed{...}" from `unique_answers`.
    # TODO: Compose `verify_prompt` with the problem and options.
    # TODO: Call `get_api_response(verify_prompt)` and strip whitespace → `chosen`.
    # TODO: Return `extract_answer(chosen)` to normalize formatting.
    # Deduplicate outputs by extracting boxed answers
    unique_answers = []
    seen_answers = set()
    for output in outputs:
        answer = extract_answer(output)
        if answer is None:
            answer = "<no_boxed_answer>"
        if answer not in seen_answers:
            seen_answers.add(answer)
            unique_answers.append(answer)

    if not unique_answers:
        return None

    # Build options string
    options = "\n".join(f"{i+1}. \\boxed{{{ans}}}" for i, ans in enumerate(unique_answers))

    # Compose verification prompt
    verify_prompt = f"""Given the math problem:
        {problem}

        And these candidate answers:
        {options}

        Which answer is correct? Reply with only the boxed answer (e.g., \\boxed{{your_answer}})."""

    chosen = get_api_response(verify_prompt).strip()
    return extract_answer(chosen)

def best_of_n_response(
    problem: str,
    N: int = 5,
    use_logprob: bool = True,
    model: str = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    port: int = 8000,
    # temp: float = 0.2
    temp: float = 0.12
) -> Optional[str]:
    """
    Run N samples on your VLLM server, then:
    - if use_logprob: pick the candidate with highest avg log-prob
    - else: hand off all N outputs to GPT to choose the best boxed answer
    """
    url = f"http://localhost:{port}/v1/chat/completions"
    prompt = SYSTEM_PROMPT + "\n" + problem

    samples = []
    for _ in range(N):
        # TODO: Build the `payload` dict with model, messages, max_tokens, temperature, and logprobs.
        # TODO: POST to `requests.post(url, json=payload)` and parse `.json()` → `resp`.
        # TODO: Extract `text` from `resp['choices'][0]['message']['content']`.
        # TODO: Compute `avg_lp` by collecting all `choice['logprobs']['content'][*]['logprob']` values.
        # TODO: Append `{"text": text, "avg_lp": avg_lp}` to `samples`.
        payload = {
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
            # "max_tokens": 500,
            "max_tokens": 3000,
            "temperature": temp,
            "logprobs": True,
            "top_logprobs": 5 # can comment this also
        }
        resp = requests.post(url, json=payload).json()
        text = resp['choices'][0]['message']['content'].strip()

        # Calculate average logprob
        logprobs = [token['logprob'] for token in resp['choices'][0]['logprobs']['content']]
        avg_lp = sum(logprobs) / len(logprobs) if logprobs else 0

        samples.append({"text": text, "avg_lp": avg_lp})


    if use_logprob:
        # TODO: Select the `sample` with the highest `avg_lp`.
        # TODO: Return `extract_answer(best["text"])`.
        # pass
        # best = max(samples, key=lambda x: x["avg_lp"])
        best = max(samples, key=lambda x: x['avg_lp'] if x['avg_lp'] is not None else float('-inf'))
        return extract_answer(best["text"])
    else:
        # TODO: Gather `outs = [s["text"] for s in samples]`.
        # TODO: Return `verify_with_gpt(problem, outs)`.
        # pass
        outs = [s["text"] for s in samples]
        return verify_with_gpt(problem, outs)


# Evaluate best of n

* modify response generation part to evalute this method.

In [ ]:
def evaluate_best_of_n(use_logprob: bool = True, N: int = 3):
    os.makedirs("results", exist_ok=True)
    results_file = (
        "evaluation_results_math500_deepseek_best_of_n_logprob.json"
        if use_logprob else
        "evaluation_results_math500_deepseek_best_of_n_gpt.json"
    )

    dataset = load_math500_dataset()
    existing = load_existing_results(results_file)
    seen = {r['index'] for r in existing}
    correct = 0

    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in seen or idx >= 30:
            continue

        prob = item['problem']
        true_ans = extract_answer(item['solution'])
        pred_ans = best_of_n_response(prob, N=N, use_logprob=use_logprob)
        is_corr = compare_answers(true_ans, pred_ans)

        save_result(results_file, {
            "index": idx,
            "problem": prob,
            "correct_answer": true_ans,
            "predicted_answer": pred_ans,
            "is_correct": is_corr
        })
        if is_corr:
            correct += 1
        print(f"corrects: {correct} / {idx+1}")

    analyze_results(load_existing_results(results_file))


In [ ]:
# !pip install --upgrade openai -q
# !pip install openai==0.28 -q
# !openai migrate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.3/661.3 kB 36.5 MB/s eta 0:00:00



Analyzing Finding files                                                         
Processed 0 files and found 0 matches


In [ ]:

# ─── Example calls ─────────────────────────────────────────────────────────
# evaluate_best_of_n(use_logprob=True,  N=500)
# evaluate_best_of_n(use_logprob=False, N=500) # the given accuracy: Accuracy: 46.67% (14/30)

print("\t<<  best of n using logprobs  >>\n")
evaluate_best_of_n(use_logprob=True,  N=3)  # 20.0% prev: you can change hyperparams or prompts to get better results
print("\t<<  best of n using LLM verifier  >>\n")
evaluate_best_of_n(use_logprob=False, N=3)
# higher N with higher temperature gives better results but takes toooo long!!! (gemeni 2.0 flash was better than 2.5 flash)

	<<  best of n using logprobs  >>



Evaluating problems:   0%|          | 1/500 [00:23<3:12:17, 23.12s/it]

corrects: 1 / 1


Evaluating problems:   0%|          | 2/500 [00:49<3:27:57, 25.05s/it]

corrects: 1 / 2


Evaluating problems:   1%|          | 3/500 [01:15<3:30:45, 25.44s/it]

... (111 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [14:08<4:04:39, 31.17s/it]

corrects: 19 / 29


Evaluating problems: 100%|██████████| 500/500 [14:41<00:00,  1.76s/it] 

corrects: 20 / 30

=== Results Summary ===
Total problems: 30
Correct answers: 20
Accuracy: 66.67%

=== Incorrect Problems ===
Problem 4:
Expected: \text{Evelyn}
Predicted: <no_boxed_answer>
---
Problem 9:
Expected: 4
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: None
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 25:
Expected: 1,-2
Predicted: None
---
Problem 26:
Expected: 144
Predicted: None
---
Problem 28:
Expected: -2 + 7i
Predicted: -2+7i
---


## Beam Search

This cell implements a beam search strategy for generating candidate reasoning chains. The method generates multiple continuations at each reasoning step, scoring each candidate based on its average token log-likelihood. By retaining and expanding only the top candidates, the approach efficiently searches for the most promising chain-of-thought that leads to the final answer in the required format.

**Key Components:**

- **Model Invocation & Token Scoring:**  
  The `call_qwen_model_raw` function sends requests to a local Qwen model endpoint using step-specific prompts. It returns generated text together with the average token log-probability, which is used as a quality metric.

- **Candidate Representation:**  
  The `BeamCandidate` class encapsulates a reasoning chain. It stores the generated text (sequence), cumulative log-probability, per-step scores, token count, and a finished flag (indicating if the candidate contains the final answer).

- **Step-wise Reasoning Generation:**  
  The `generate_reasoning_steps` function creates multiple candidate continuations for each reasoning step. Different prompts guide the generation for understanding the problem, planning a strategy, and producing the final answer (which is always enclosed in a `\boxed{}` block).

- **Beam Search Process:**  
  The `beam_search` function expands candidate chains over several steps. At each step, candidates are updated by appending the new reasoning text and averaging the log-probabilities from all tokens(you can use num_token now). Only the top candidates (based on cumulative score) are retained for further expansion.

- **Final Answer Extraction:**  
  The `run_qwen_beam_search` function initializes the prompt with the problem statement, runs the beam search, and extracts the final answer from the best candidate if it is complete.

This structured approach ensures efficient exploration of possible reasoning paths while focusing on the most promising ones to arrive at the final answer in the expected format.


In [ ]:
## I added this section to be more readable :)
step_suffixes = {
    1: "Let's understand the problem.", # understand
    2: "Let's plan how to solve it.", # plan
    3: "Let's solve it and put the final answer within \\boxed{}." # solve
}

In [ ]:
import os
import requests
from typing import Optional, List
from tqdm import tqdm

def score_with_gpt(problem: str, reasoning_step: str) -> float:
    """
    Ask a high‑quality LLM (via get_api_response) to rate the given
    reasoning step on a 0–1 scale. Returns the numeric score.
    """
    prompt = (
        "You are a rigorous math reasoning evaluator.\n\n"
        f"Problem:\n{problem}\n\n"
        "Candidate reasoning step:\n"
        f"\"\"\"\n{reasoning_step}\n\"\"\"\n\n"
        "On a scale from 0 (completely incorrect) to 1 (perfectly correct), "
        "rate how valid and useful this step is toward solving the problem. "
        "Reply with only a number between 0 and 1."
    )
    # TODO: Call get_api_response(prompt), strip the result
    # TODO: Parse float(resp), fallback to 0.0 on ValueError, and return it
    response = get_api_response(prompt).strip()
    try:
        score = float(response)
        return max(0.0, min(1.0, score))
    except ValueError:
        return 0.0

def call_qwen_model_raw(prompt: str, step_num: int, temperature: float = 0.3):
    """
    Sends a request to the local Qwen endpoint and returns the generated text
    along with the average token log-probability and token count.
    """
    max_tokens = {1: 500, 2: 800, 3: 1700}.get(step_num, 500)
    url = "http://localhost:8000/v1/chat/completions"
    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "logprobs": True,
    }
    # TODO: resp = requests.post(url, json=payload).json()
    # TODO: Extract text = resp['choices'][0]['message']['content'].strip()
    # TODO: Gather token_logprobs from resp['choices'][*]['logprobs']['content'][*]['logprob']
    # TODO: Compute avg_logprob and num_token, then return (text, avg_logprob, num_token)
    resp = requests.post(url, json=payload).json()
    text = resp['choices'][0]['message']['content'].strip()
    # token_logprobs = resp['choices'][0]['logprobs']
    token_logprobs = [choice['logprob'] for choice in resp['choices'][0]['logprobs']['content']]
    num_token = len(token_logprobs)
    # print("token_logprobs", token_logprobs)
    avg_logprob = sum(token_logprobs) / num_token if num_token > 0 else 0.0
    return text, avg_logprob, num_token

class BeamCandidate:
    def __init__(
        self,
        sequence: str,
        cumulative_log_prob: float,
        step_scores: List[float],
        finished: bool,
        num_token: int
    ):
        self.sequence = sequence
        self.cumulative_log_prob = cumulative_log_prob
        self.step_scores = step_scores
        self.finished = finished
        self.num_token = num_token

    def __repr__(self):
        return (
            f"BeamCandidate(score={self.cumulative_log_prob:.3f}, "
            f"finished={self.finished}, sequence=[...])"
        )

def generate_reasoning_steps(
    context: str,
    step_num: int,
    top_k: int,
    use_logprob: bool = True,
    problem: Optional[str] = None
):
    """
    Generate top_k candidate continuations for the current reasoning step.
    """
    candidates = []
    for i in range(top_k):
        # TODO: Build `suffix` based on step_num (1=understand, 2=plan, 3=solve)
        # TODO: prompt = context + suffix
        # TODO: output, avg_token_prob, num_token = call_qwen_model_raw(prompt, step_num)
        # TODO: if use_logprob: score = avg_token_prob else: assert problem, score = score_with_gpt(problem, output)
        # TODO: finished = "\\boxed{" in output
        # TODO: candidates.append((output.strip(), score, num_token, finished))
        suffix = step_suffixes.get(step_num, "")
        prompt = context + "\n" + suffix
        output, avg_token_prob, num_token = call_qwen_model_raw(prompt, step_num)
        if use_logprob:
            score = avg_token_prob
        else:
            assert problem is not None, "Problem must be provided for GPT scoring"
            score = score_with_gpt(problem, output)
        finished = "\\boxed{" in output
        candidates.append((output.strip(), score, num_token, finished))

    return candidates

def beam_search(
    init_problem_prompt: str,
    beam_width: int = 3,
    max_steps: int = 3,
    top_k: int = 2,
    use_logprob: bool = True
):
    """
    Beam search over reasoning steps. If use_logprob=False, uses GPT verifier
    to score each node instead of token log-prob.
    """
    # TODO: Extract `problem` from init_problem_prompt
    # TODO: initial = BeamCandidate(sequence=init_problem_prompt, cumulative_log_prob=0.0, step_scores=[], finished=False, num_token=0)
    # TODO: beams = [initial]
    problem = init_problem_prompt.split('\n')[-1] # (assuming it's the last line)
    initial = BeamCandidate(sequence=init_problem_prompt, cumulative_log_prob=0.0, step_scores=[], finished=False, num_token=0)
    beams = [initial]

    for step in range(1, max_steps + 1):
        new_beams = []
        # TODO: For each cand in beams:
        #           if cand.finished: new_beams.append(cand); continue
        #           step_cands = generate_reasoning_steps(cand.sequence, step, top_k, use_logprob, problem)
        #           For each (text, score, n_tok, finished) in step_cands:
        #               seq = cand.sequence + "\n" + text
        #               total_tokens = cand.num_token + n_tok
        #               cum = ((cand.cumulative_log_prob * cand.num_token) + score * n_tok) / total_tokens
        #               new_beams.append(BeamCandidate(seq, cum, cand.step_scores + [score], finished, total_tokens))
        #       Sort new_beams by cumulative_log_prob desc and slice to beam_width
        #       beams = new_beams
        #       If all beams finished: break
        for cand in beams:
            if cand.finished:
                new_beams.append(cand)
                continue
            step_cands = generate_reasoning_steps(cand.sequence, step, top_k, use_logprob, problem)
            for text, score, n_tok, finished in step_cands:
                seq = cand.sequence + "\n" + text
                total_tokens = cand.num_token + n_tok
                cum = ((cand.cumulative_log_prob * cand.num_token) + score * n_tok) / total_tokens if total_tokens > 0 else 0.0
                new_cand = BeamCandidate(
                    sequence=seq,
                    cumulative_log_prob=cum,
                    step_scores=cand.step_scores + [score],
                    finished=finished or cand.finished,
                    num_token=total_tokens
                )
                new_beams.append(new_cand)
        new_beams.sort(key=lambda x: x.cumulative_log_prob, reverse=True)
        beams = new_beams[:beam_width]

        # Early exit if all beams are finished
        if all(b.finished for b in beams):
            break

    # TODO: finished = [b for b in beams if b.finished]
    # TODO: best = max(finished, key=...) if finished else beams[0]
    # TODO: return best
    finished = [b for b in beams if b.finished]
    if finished:
        return max(finished, key=lambda b: b.cumulative_log_prob)
    else:
        return max(beams, key=lambda b: b.cumulative_log_prob)

def run_qwen_beam_search(
    problem: str,
    beam_width: int,
    max_steps: int,
    top_k: int,
    log_level,
    use_logprob: bool = True
):
    """
    Performs beam search and extracts the final boxed answer.
    """
    # TODO: prompt = f"Consider this problem:\n{problem}"
    # TODO: best = beam_search(prompt, beam_width, max_steps, top_k, use_logprob)
    # TODO: if best.finished:
    #           ans = extract_answer(best.sequence)
    #           print(f"\nExtracted Final Answer: {ans}")
    #           return ans
    #       else:
    #           print("No final answer found.")
    #           return None
    init_prompt = f"Consider this problem:\n{problem}"
    best = beam_search(init_prompt, beam_width, max_steps, top_k, use_logprob)
    if best.finished:
        ans = extract_answer(best.sequence)
        print(f"\nExtracted Final Answer: {ans}")
        return ans
    else:
        print("No final answer found.")
        return None

# Evaluate beam search
* modify response generation part to evalute this method.

In [ ]:
def evaluate_beam_search(use_logprob: bool = True):
    """
    Evaluate beam search on MATH‑500, toggling between log‑prob scoring
    and GPT/Gemini–based verification for each reasoning node.
    """
    os.makedirs("results", exist_ok=True)

    suffix = "logprob" if use_logprob else "gpt"
    results_file = f"evaluation_results_math500_deepseek_beam_search_{suffix}.json"

    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {r['index'] for r in existing_results}

    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes or idx >= 30:
            continue

        problem_text   = item['problem']
        correct_answer = extract_answer(item['solution'])

        # Run beam search with the desired scoring method
        response = run_qwen_beam_search(
            problem     = problem_text,
            beam_width  = 3,
            max_steps   = 3,
            top_k       = 2,
            log_level   = 1,              # existing parameter
            use_logprob = use_logprob     # True = token log‑prob, False = GPT verifier
        )
        predicted_answer = response

        is_correct = compare_answers(correct_answer, predicted_answer)
        save_result(results_file, {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        })

        if is_correct:
            cnt += 1
        print(f"corrects: {cnt} / {idx+1}")

    final_results = load_existing_results(results_file)
    analyze_results(final_results)


In [ ]:
# Example usage:
# I copied from above and ran with print("\t<< best of n using logprobs  >>\n") but the currect one is print("\t<<  using logprobs  >>\n").
# Because it took a long time to rerun I woudlnt run it again :)
print("\t<<  using logprobs  >>\n")
evaluate_beam_search(use_logprob=True)   # pick by avg token log‑prob
print("\t<<  LLM verifier  >>\n")
evaluate_beam_search(use_logprob=False)  # pick by GPT/Gemini verification
# given accuracy: 66.67% (20/30)

	<<  best of n using logprobs  >>



Evaluating problems:   0%|          | 1/500 [01:13<10:08:13, 73.13s/it]


Extracted Final Answer: (3, \frac{\pi}{2})
corrects: 1 / 1


Evaluating problems:   0%|          | 2/500 [05:20<24:15:55, 175.41s/it]


Extracted Final Answer: p - q
corrects: 2 / 2


Evaluating problems:   1%|          | 3/500 [07:10<20:07:16, 145.75s/it]

... (107 outputs left out)

Evaluating problems:   5%|▌         | 27/500 [1:41:02<33:52:25, 257.81s/it]

No final answer found.
corrects: 16 / 27


Evaluating problems:   6%|▌         | 28/500 [1:42:30<27:06:47, 206.80s/it]


Extracted Final Answer: 78
corrects: 17 / 28


In [ ]:
# my runtime disconnected in the previous cell :)
print("\t<<  LLM verifier  >>\n")
evaluate_beam_search(use_logprob=False)  # pick by GPT/Gemini verification

	<<  LLM verifier  >>



Evaluating problems:   0%|          | 1/500 [02:45<22:56:00, 165.45s/it]


Extracted Final Answer: \left(3, \dfrac{\pi}{2}\right)
corrects: 1 / 1


Evaluating problems:   0%|          | 2/500 [07:07<30:46:03, 222.42s/it]


Extracted Final Answer: p - q
corrects: 2 / 2


Evaluating problems:   1%|          | 3/500 [11:04<31:35:22, 228.82s/it]

... (51 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [1:48:45<28:45:30, 219.81s/it]


Extracted Final Answer: -2 + 7i
corrects: 18 / 29


Evaluating problems: 100%|██████████| 500/500 [1:51:03<00:00, 13.33s/it]   


Extracted Final Answer: 225
corrects: 19 / 30

=== Results Summary ===
Total problems: 30
Correct answers: 19
Accuracy: 63.33%

=== Incorrect Problems ===
Problem 9:
Expected: 4
Predicted: None
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: None
---
Problem 17:
Expected: \pi
Predicted: None
---
Problem 18:
Expected: 28
Predicted: None
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: None
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 25:
Expected: 1,-2
Predicted: 1
---
Problem 26:
Expected: 144
Predicted: None
---
Problem 28:
Expected: -2 + 7i
Predicted: -2 + 7i
---


## Self-Refinement

This cell implements a self-refinement approach to solving math problems. Initially, it generates a solution using a fixed system prompt that enforces a step-by-step reasoning process and a final answer format enclosed in `\boxed{}`. Then, through iterative feedback, the model is asked to analyze its own output and refine it if necessary. This loop ensures that the final answer is both correct and clearly formatted.


In [ ]:
import re
import requests

SYSTEM_PROMPT = '''You are solving mathematics problems.

Please think step by step.

Important: Always end your solution with the final answer in this format:

\\[
\\boxed{your_answer_here}
\\]

The entire answer should be contained completely within the \\boxed{} command.'''

def generate_content(prompt: str) -> str:
    """
    Sends `prompt` to the local Qwen endpoint and returns the generated text.
    """
    url = "http://localhost:8000/v1/chat/completions"
    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 2000,
        "temperature": 0.3,
        # "temperature": 0.1, # get better results probably
    }
    # TODO: Send HTTP POST to `url` with `payload`, parse the JSON response
    # TODO: Extract `content` from `resp['choices'][0]['message']['content']` and strip whitespace
    # TODO: Return the resulting string
    # pass
    resp = requests.post(url, json=payload).json()
    text = resp['choices'][0]['message']['content'].strip()
    return text

def self_refine(problem: str, max_iter: int = 2) -> Optional[str]:
    """
    Iteratively refines the model’s output on `problem` using feedback loops.
    """
    # TODO: Build initial `prompt` by concatenating SYSTEM_PROMPT and `problem`
    # TODO: Call `generate_content(prompt)` to get `current_output`
    prompt = SYSTEM_PROMPT + "\n" + problem
    current_output = generate_content(prompt)

    for iteration in range(max_iter):
        # TODO: Construct `feedback_prompt` that includes:
        #         - the original `prompt`
        #         - the `current_output`
        #         - instructions to output [FEEDBACK] and [REFINEMENT_NEEDED]
        # TODO: Call `generate_content(feedback_prompt)` → `feedback_response`
        # TODO: Use `re.search` with pattern r"\[(?i:feedback)\](.*?)\[(?i:refinement_needed)\](.*)"
        #       to extract `feedback` and `refinement_flag`
        # TODO: Determine `refinement_needed` (default True, set False if flag is "no" or feedback contains stop phrases)
        # TODO: If `not refinement_needed`: break out of loop

        # TODO: Build `refine_prompt` that includes:
        #         - the original `prompt`
        #         - the `current_output`
        #         - the extracted `feedback`
        # TODO: Call `generate_content(refine_prompt)` → `refined_output`
        # TODO: If `refined_output.strip() == current_output.strip()`: break
        # TODO: Otherwise, set `current_output = refined_output`
        feedback_prompt = f"""
            You have generated the following output for the given prompt:

            Prompt:
            {prompt}

            Your output:
            {current_output}

            Now, please provide feedback on your output. First, output [FEEDBACK] followed by your thoughts on whether the output is correct and complete. Then, output [REFINEMENT_NEEDED] followed by "yes" if you think the output needs further refinement, or "no" if you think it's already correct and complete.
        """
        feedback_response = generate_content(feedback_prompt)
        match = re.search(r"\[(?i:feedback)\](.*?)\[(?i:refinement_needed)\](.*)", feedback_response, re.DOTALL)
        if match:
            feedback = match.group(1).strip()
            refinement_flag = match.group(2).strip().lower()
            refinement_needed = refinement_flag == "yes"
        else:
            feedback = ""
            refinement_needed = True

        if not refinement_needed:
            break

        refine_prompt = f"""
            Based on the feedback, please refine your output.

            Original prompt:
            {prompt}

            Original output:
            {current_output}

            Feedback:
            {feedback}

            Please provide a refined output that addresses the feedback.
        """
        refined_output = generate_content(refine_prompt)
        if refined_output.strip() == current_output.strip():
            break
        current_output = refined_output

    # TODO: Use `extract_answer(current_output)` to get the final boxed answer
    # TODO: Return that answer (or None if no boxed answer found)
    # pass
    return extract_answer(current_output)


# Evaluate self refiner
* modify response generation part to evalute this method.

In [ ]:
def evaluate_self_refiner():
    os.makedirs("results", exist_ok=True)
    results_file = "evaluation_results_math500_deepseek_self_refiner.json"
    dataset = load_math500_dataset()
    existing_results = load_existing_results(results_file)
    processed_indexes = {result['index'] for result in existing_results}
    cnt = 0
    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed_indexes :
            continue
        if idx >= 30:
          break
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])
        ##########################################################
        response = self_refine(problem_text,3)
        predicted_answer = response
        ##########################################################
        is_correct = compare_answers(correct_answer, predicted_answer)
        result = {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        }
        save_result(results_file, result)
        if is_correct:
          cnt += 1
        print(f"corrects :  {cnt} idx: {idx}")
    final_results = load_existing_results(results_file)
    analyze_results(final_results)

In [ ]:
evaluate_self_refiner() # given accuracy: Accuracy: 62.50% (20/32)

Evaluating problems:   0%|          | 1/500 [01:16<10:37:39, 76.67s/it]

corrects :  1 idx: 0


Evaluating problems:   0%|          | 2/500 [03:19<14:20:58, 103.73s/it]

corrects :  2 idx: 1


Evaluating problems:   1%|          | 3/500 [04:57<13:56:25, 100.98s/it]

corrects :  3 idx: 2


... (50 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [54:04<12:46:39, 97.66s/it] 

corrects :  16 idx: 28


Evaluating problems:   6%|▌         | 30/500 [55:02<14:22:23, 110.09s/it]

corrects :  17 idx: 29

=== Results Summary ===
Total problems: 30
Correct answers: 17
Accuracy: 56.67%

=== Incorrect Problems ===
Problem 9:
Expected: 4
Predicted: None
---
Problem 10:
Expected: 2220
Predicted: 0
---
Problem 11:
Expected: \frac{3}{56}
Predicted: None
---
Problem 12:
Expected: 284
Predicted: None
---
Problem 17:
Expected: \pi
Predicted: 0
---
Problem 20:
Expected: 6+9i
Predicted: 6 + 9i
---
Problem 21:
Expected: 13535
Predicted: None
---
Problem 22:
Expected: 5
Predicted: None
---
Problem 23:
Expected: x=5
Predicted: 5
---
Problem 24:
Expected: 10
Predicted: 22\%
---
Problem 25:
Expected: 1,-2
Predicted: 1
---
Problem 26:
Expected: 144
Predicted: None
---
Problem 28:
Expected: -2 + 7i
Predicted: -2 + 7i
---


# 🚀 Overview (Part 2): Implementing A*, Monte Carlo Tree Search (MCTS), and Tree of Thoughts (ToT)

Ok so now every thing is ready to start part 2, This part aims to explore three sophisticated search and reasoning algorithms—**A\***, **Monte Carlo Tree Search (MCTS)**, and **Tree of Thoughts (ToT)**—to solve challenging mathematical problems, specifically using the MATH-500 dataset and an LLM (Language Model). Before diving into the implementation, we provide a comprehensive overview of each algorithm, highlighting their core mechanisms, practical considerations, and potential challenges.

---

## 🌟 1. A* Search Algorithm

**A*** is an informed search algorithm designed for efficiently finding the shortest path or optimal solution in a search space using heuristics.

### 🔹 Core Principles:
- **Best-first search:** Expands nodes based on a cost function, \(f(n)\), prioritizing paths that seem closer to a goal.
- **Heuristic evaluation:** Uses a heuristic function \(h(n)\) to estimate the cost from the current node to the goal.

### 🔹 Components:
- **Cost function \(f(n) = g(n) + h(n)\)**, where:
  - \(g(n)\): Actual cost from start node to node \(n\).
  - \(h(n)\): Estimated cost from node \(n\) to the goal (heuristic).

### 🔹 Practical Considerations:
- Heuristic function must be **efficient** and **accurate**.
- A good heuristic drastically reduces computation time and search complexity.

### ⚠️ Challenges in Implementation:
- **Designing an effective heuristic:**
  - Challenge to accurately estimate "distance" from partial reasoning steps to the solution.
  - Requires leveraging language models to score plausibility.
- **Computational efficiency:**
  - Heuristic evaluation via LLM queries could be computationally costly if not managed carefully.
- **Admissibility and consistency:**
  - Ideally, heuristic must be admissible (never overestimates the true cost) to guarantee optimality.

---

## 🎲 2. Monte Carlo Tree Search (MCTS)

**MCTS** is a probabilistic algorithm widely used in decision-making scenarios, especially effective in complex problems with uncertain outcomes, such as mathematical reasoning guided by language models.

### 🔹 Core Principles:
MCTS explores decision trees using **randomized simulations** (rollouts) and statistical sampling.

**Four main steps in MCTS**:
1. **Selection**: Uses UCT (Upper Confidence Bound) to balance exploration and exploitation.
2. **Expansion**: Adds new unexplored nodes to the tree.
3. **Simulation**: Conducts rollouts from newly expanded nodes to estimate potential outcomes.
4. **Backpropagation**: Updates statistical measures based on simulation results.

### 🔹 Components:
- **UCT formula** for selection:
  $$
  \text{UCT} = \frac{w_i}{n_i} + C\sqrt{\frac{\ln N}{n_i}}
  $$
  - \($w_i$\): Total rewards.
  - \($n_i$\): Visits to node \($i$\).
  - \($N$\): Visits to parent node.
  - \($C$\): Exploration constant (usually \($\sqrt{2}$\)).

- **Rollout (simulation)**:
  - Typically involves letting the LLM complete reasoning steps to the end, evaluating correctness.

### 🔹 Practical Considerations:
- Effective rollout policies significantly impact accuracy and efficiency.
- Balance exploration (testing new reasoning paths $(\frac{w_i}{n_i})$) and exploitation (refining known good solutions $(C\sqrt{\frac{\ln N}{n_i}})$).

### ⚠️ Challenges in Implementation:
- **Computational overhead**:
  - Running multiple LLM-based rollouts per node can be slow and computationally expensive.
- **Optimal parameter tuning**:
  - Choosing the exploration constant \(C\) and number of simulations impacts performance significantly.
- **Quality of simulation outcomes**:
  - Poor rollout outcomes (random or inaccurate completions) can misguide the search tree.

---

## 🌳 3. Tree of Thoughts (ToT)

**Tree of Thoughts (ToT)** is specifically designed for structured reasoning tasks with language models, extending their capabilities through explicit evaluation and pruning of reasoning paths.

### 🔹 Core Principles:
- Generate multiple candidate "thoughts" (reasoning paths).
- Evaluate each thought explicitly (often through LLM-based scoring or consistency checking).
- Iteratively prune weaker reasoning paths, keeping the most promising solutions.

### 🔹 Components:
- **Thought generation**: Multiple candidate reasoning steps generated at each node.
- **Thought evaluation**: Explicit scoring (via LLMs) to judge mathematical correctness or plausibility.
- **Pruning**: Remove less promising reasoning branches based on evaluation.

### 🔹 Practical Considerations:
- Explicit node evaluation adds a structured layer of reasoning not present in simpler methods.
- Allows LLMs to reason more deliberately by systematically exploring and eliminating alternatives.

### ⚠️ Challenges in Implementation:
- **Evaluation complexity**:
  - Frequent explicit evaluations by LLM can slow the process.
  - Requires efficient prompting and scoring techniques.
- **Self-consistency**:
  - Maintaining logical consistency across multiple branches can be difficult, especially for complex math problems.
- **Scaling**:
  - Managing multiple branches of reasoning can quickly become computationally expensive without careful control.

---


**Next, we will begin the practical implementation step-by-step.**


# Implementing Tree of Thoughts (ToT)

Before writing any code, it’s essential to map out the steps and functions we need for our Tree of Thoughts (ToT) implementation. In ToT, we aren’t just following one linear chain of reasoning but instead generating a tree of candidate reasoning paths (“thoughts”), evaluating them, and then expanding the most promising ones. We can leverage our existing helper functions (such as those for extracting and normalizing answers) as part of the evaluation process.

Below is an outline of the steps and functions we’ll need:

---

## 1. **Node Representation**

We need a way to represent each node in our reasoning tree. A node could include:
- **Current reasoning text**: The partial solution or thought generated so far.
- **Candidate thoughts**: A list of potential next steps (children nodes).
- **Evaluation score**: A score indicating how promising the node is (based on plausibility or correctness).
- **Metadata**: Such as depth in the tree or a reference to the parent node.

**Potential Functions/Classes:**
- `class Node`: A class that encapsulates the above properties.
- `add_child(self, child_node)`: A method to attach a new candidate thought.

---

## 2. **Candidate Thought Generation**

This function will use the LLM (via our `get_llm_response` function) to generate multiple candidate reasoning steps given a node's current state.

**Key Points:**
- The prompt should be carefully crafted to ask the LLM for alternative reasoning steps.
- We can use techniques such as few-shot prompting to guide the LLM in generating diverse thoughts.

**Potential Function:**
- `def generate_candidate_thoughts(node: Node, num_candidates: int) -> List[str]:`
  - This function takes the current reasoning state from `node` and returns a list of candidate thoughts (as strings).

---

## 3. **Candidate Thought Evaluation**

Once we have multiple candidate thoughts, we need to score them. The evaluation could be based on:
- **Model’s self-assessment:** Ask the LLM to rate each candidate on a scale (e.g., 1 to 10) for mathematical plausibility or correctness.
- **Heuristics based on helper functions:** Use helper functions like `extract_answer` and `normalize_answer` to check whether a candidate thought moves closer to a correct answer or simplifies the expression.

**Potential Function:**
- `def evaluate_candidate_thought(candidate: str) -> float:`
  - This function might prompt the LLM with the candidate reasoning step, asking, “How plausible or correct is this step?” and return a numeric score.
  - Alternatively, it might combine an LLM score with our own heuristic checks.

---

## 4. **Pruning and Selection**

After evaluating the candidates, we must select the most promising ones for further expansion. Pruning involves:
- Ranking candidate thoughts by their evaluation score.
- Keeping only the top N candidates (to control the tree size).

**Potential Function:**
- `def select_best_candidates(candidates: List[str], scores: List[float], top_n: int) -> List[str]:`
  - This function will combine the candidate list and their scores to select the best ones for further exploration.

---

## 5. **Node Expansion**

For each node, the process is:
1. **Generate candidate thoughts** using the generation function.
2. **Evaluate each candidate** using the evaluation function.
3. **Select the best candidate(s)** using the pruning/selection function.
4. **Expand the tree** by creating child nodes for each selected candidate.

**Potential Function:**
- `def expand_node(node: Node, num_candidates: int, top_n: int) -> None:`
  - This function integrates candidate generation, evaluation, and pruning to add new child nodes to the given `node`.

---

## 6. **Stopping Criteria**

We need clear criteria for when to stop expanding the tree:
- **Complete solution found:** When a node contains a complete solution (e.g., using `extract_answer` to verify that a boxed answer exists).
- **Depth or resource limits:** When a maximum depth is reached or computational resources are constrained.

**Potential Function:**
- `def is_solution(node: Node) -> bool:`
  - This function checks if a node’s reasoning contains a valid, complete answer.
- `def stop_expansion(node: Node, max_depth: int) -> bool:`
  - This function checks if the node has reached the maximum allowed depth.

---

## 7. **Integration with Helper Functions**

Our previous helper functions play a crucial role in the ToT implementation:
- **Extracting and normalizing answers:**  
  Use `extract_answer` and `normalize_answer` to interpret candidate outputs and compare them against the expected solution.
- **Comparison functions:**  
  Use `compare_answers` to help decide if a candidate thought is moving in the right direction.
- **LLM response function:**  
  `get_llm_response` is used both for generating candidate thoughts and possibly for scoring them.

---

## 8. **Overall ToT Flow**

Putting it all together, here is an outline of the overall ToT process:
1. **Initialize the root node** with the initial problem statement.
2. **While the stopping criteria are not met:**
   - For the current node, generate candidate thoughts.
   - Evaluate each candidate.
   - Select and expand the best candidates to form new child nodes.
3. **Once a candidate thought leads to a complete solution:**
   - Use helper functions to verify correctness.
   - Return or record the successful reasoning path.

---

In [ ]:
import requests

def get_llm_response(prompt):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    # "max_tokens": 500,
    "max_tokens": 2000,
    "temperature": 0.6
    }
    response = requests.post(url, json=payload)
    # return response.json()['choices'][0]['message']['content'].strip()
    res_json = response.json()
    if 'choices' in res_json:
        return res_json['choices'][0]['message']['content'].strip()
    else: # error: res_json -> {'object': 'error', 'message': "This model's maximum context length is 5192 tokens. However, you requested 6142 tokens (3142 in the messages, 3000 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400}
        # print("=====================")
        # print(res_json)
        # print("=====================")
        return ""

In [ ]:
import re
import time
from typing import List, Optional

def score_with_gpt(problem: str, candidate: str) -> float:
    """
    Ask a high‑quality LLM (via get_api_response) to rate the given
    final answer on a scale from 1 (poor) to 10 (excellent).
    """
    # TODO: build `eval_prompt` string using `problem` and `candidate`
    # resp = get_api_response(eval_prompt).strip()
    # TODO: use `re.search` to extract the first numeric score from `resp`
    # return float(match) or 0.0 on failure

    ### hint: designing better prompt can make the result better! ###
    eval_prompt = (
        "You are a precise mathematics grader. "
        f"Problem: {problem}\n\n"
        f"Candidate solution: {candidate}\n\n"
        "On a scale from 1 (completely incorrect) to 10 (perfectly correct), "
        "how accurate is this solution? Reply with only a number between 1 and 10."
    )
    # eval_prompt = f"""You are a math problem solver. Given this problem:
    #     {problem}

    #     Candidate solution step:
    #     {candidate}

    #     Rate the mathematical correctness and progress towards solution from 0 (completely wrong) to 10 (perfectly correct). Reply with only a number between 0 and 10."""

    resp = get_api_response(eval_prompt).strip()
    # match = re.search(r'\d+', resp)
    match = re.search(r'\d+(?:\.\d+)?', resp)
    # return float(match.group()) if match else 0.0
    return max(10.0, min(1.0, float(match.group()))) if match else 0.0

class Node:
    def __init__(self, state: str, depth: int = 0, parent: Optional['Node'] = None):
        self.state = state
        self.depth = depth
        self.parent = parent
        self.children: List[Node] = []      # easy: list to hold child nodes
        self.score: Optional[float] = None  # will be set once evaluated

    def add_child(self, child_node: 'Node'):
        self.children.append(child_node)

    def __repr__(self):
        # easy: show depth, score, and a truncated preview of `state`
        preview = self.state if len(self.state) < 50 else self.state[:47] + "..."
        return f"Node(depth={self.depth}, score={self.score}, state='{preview}')"

class TreeOfThoughts:
    def __init__(
        self,
        num_candidates: int = 3,
        top_n: int = 2,
        max_depth: int = 3,
        verbose: bool = False
    ):
        self.num_candidates = num_candidates
        self.top_n = top_n
        self.max_depth = max_depth
        self.verbose = verbose
        self.root_state: Optional[str] = None

    def generate_candidate_thoughts(self, node: Node) -> List[str]:
        """
        Prompt the LLM to produce `num_candidates` boxed answers for `node.state`.
        """
        # TODO: compose prompt with node.state
        # TODO: call `get_llm_response(prompt)`
        # TODO: split into non-empty lines and return exactly `self.num_candidates` answers

        ### hint: designing better prompt can make the result better! ###
        prompt = (
            "You are a mathematical problem solver. "
            f"Problem: {node.state}\n\n"
            f"Generate {self.num_candidates} distinct possible next steps or solutions "
            "to this problem. Each solution should end with a final answer in the LaTeX boxed format "
            "\\boxed{...}. Provide each solution on a new line."
        )
        response = get_llm_response(prompt)
        # answers = re.findall(r'\\boxed\{(.*?)\}', response) # if did not work test with answers
        # candidates = answers
        candidates = [line.strip() for line in response.split('\n') if line.strip()]

        # Ensure we have exactly num_candidates
        if len(candidates) < self.num_candidates:
            candidates += [candidates[-1]] * (self.num_candidates - len(candidates))
        return candidates[:self.num_candidates]
        # return [f"\\boxed{{{ans}}}" for ans in answers[:self.num_candidates]]

    def evaluate_candidate_thought(self, candidate: str) -> float:
        # simple wrapper around score_with_gpt
        assert self.root_state is not None
        return score_with_gpt(self.root_state, candidate)

    def select_best_candidates(self, candidates: List[str]) -> List[str]:
        # TODO: for each cand in candidates:
        #         - score = self.evaluate_candidate_thought(cand)
        #         - time.sleep(0.5)
        #       sort by score descending and return top `self.top_n`
        scored = []
        for cand in candidates:
            score = self.evaluate_candidate_thought(cand)
            time.sleep(0.5)  # Rate limit API calls
            scored.append((score, cand))

        scored.sort(reverse=True, key=lambda x: x[0])
        return [cand for score, cand in scored[:self.top_n]]

    def expand_node(self, node: Node) -> None:
        if self.verbose:
            print(f"\nExpanding depth {node.depth} state:\n{node.state}\n")
        # TODO: raw = self.generate_candidate_thoughts(node)
        # TODO: best = self.select_best_candidates(raw)
        raw = self.generate_candidate_thoughts(node)
        best = self.select_best_candidates(raw)

        # for ans in []:  # replace [] with `best`
        for ans in best:
            child = Node(state=ans, depth=node.depth + 1, parent=node)
            # TODO: child.score = self.evaluate_candidate_thought(ans)
            # TODO: if not boxed, retry up to 3 times with a strict prompt
            child.score = self.evaluate_candidate_thought(ans)

            # Ensure we have a boxed answer
            attempts = 0
            while not self.is_solution(child) and attempts < 3:
                fallback_prompt = (
                    f"Problem: {self.root_state}\n\n"
                    f"Current reasoning: {child.state}\n\n"
                    "Please provide ONLY your final answer in the exact format \\boxed{...} "
                    "with no additional explanation."
                )
                child.state = get_llm_response(fallback_prompt).strip()
                attempts += 1

                if self.verbose:
                    print(f"Fallback attempt {attempts}: new state:\n{child.state}")

            node.add_child(child)
            if self.verbose:
                print(f"Added child: {child}")

    def is_solution(self, node: Node) -> bool:
        # TODO: return True if `extract_answer(node.state)` yields non-empty
        return extract_answer(node.state) is not None

    def stop_expansion(self, node: Node) -> bool:
        return node.depth >= self.max_depth

    def search(self, root_state: str) -> Node:
        """
        Build the tree until solutions found or max depth reached.
        """
        self.root_state = root_state
        root = Node(state=root_state, depth=0)
        frontier = [root]
        while frontier:
            node = frontier.pop(0)
            if self.is_solution(node) or self.stop_expansion(node):
                continue
            self.expand_node(node)
            frontier.extend(node.children)
        return root


# Testing the Tree of Thoughts Method with Minimal Hyperparameters

This cell demonstrates a test run of the Tree of Thoughts (ToT) framework using minimal hyperparameters. The goal is to ensure that a complete final answer (in the format `\boxed{...}`) is extracted from the model's output.

**Key Steps:**

- **Instantiate TOT:**  
  The TOT instance is created with `num_candidates=1`, `top_n=1`, and `max_depth=1` in verbose mode. This minimal setup is used for quick testing.

- **Run the Search:**  
  The TOT search is executed on the sample problem:  
  *"Solve the integral: \( \int_0^1 x^2 \, dx \)"*

- **Print the TOT Tree:**  
  A recursive function (`print_tree`) prints the entire search tree, allowing inspection of each node's state and depth.

- **Extract Final Answer:**  
  All nodes in the tree are collected. If a node is found that contains a final answer (determined via the helper function `extract_answer`), then the node's state is overwritten to display only the final answer in the correct boxed format.  
  If no such node is found, a fallback prompt forces the model to output the final answer exclusively.

This setup helps verify that the TOT framework correctly isolates and formats the final answer, ensuring the result is comparable to the expected output.


In [ ]:
# Test the TreeOfThoughts with minimal settings.
# Generation of candidate answers still uses your primary LLM via get_llm_response,
# while evaluation/scoring uses only the GPT/Gemini verifier.

tot = TreeOfThoughts(num_candidates=1,
                     top_n=1,
                     max_depth=1,
                     verbose=True)

initial_problem = "Solve the integral: \\( \\int_0^1 x^2 \\, dx \\)"
tot_tree = tot.search(initial_problem)

# Helper to print the whole tree
def print_tree(node: Node, indent: str = ""):
    print(indent + repr(node))
    for child in node.children:
        print_tree(child, indent + "  ")

print_tree(tot_tree)

# Collect all nodes and find the first complete solution
def collect_all_nodes(node: Node) -> List[Node]:
    nodes = [node]
    for child in node.children:
        nodes.extend(collect_all_nodes(child))
    return nodes

all_nodes = collect_all_nodes(tot_tree)
solution_node = next((n for n in all_nodes if tot.is_solution(n)), None)

print("---")
if solution_node:
    final = extract_answer(solution_node.state)
    solution_node.state = f"\\boxed{{{final}}}"
    print("\nFinal Answer Found:")
    print(solution_node.state)
else:
    # If no solution node was produced, force a final answer via GPT verifier
    fallback_prompt = (
        f"Based on the problem: \"{initial_problem}\", please provide ONLY your final answer "
        "in the exact format \\boxed{...}."
    )
    forced = get_api_response(fallback_prompt).strip()
    final = extract_answer(forced) or forced
    forced = f"\\boxed{{{final}}}"
    print("\nNo complete final answer was found. Forcing final answer:")
    print(forced)


Expanding depth 0 state:
Solve the integral: \( \int_0^1 x^2 \, dx \)

Fallback attempt 1: new state:
Okay, so I have to solve the integral \( \int_0^1 x^2 \, dx \). Hmm, let me recall what I know about integrals. Integrals can be used to find the area under a curve, and this one is specifically from 0 to 1. The function inside the integral is \( x^2 \), which is a parabola opening upwards. I remember that the integral of \( x^n \) is \( \frac{x^{n+1}}{n+1} \) plus a constant, right? So maybe I can use that rule here.

Wait, let me make sure I'm applying it correctly. The general formula for the integral of \( x^k \) is \( \frac{x^{k+1}}{k+1} \), as long as \( k \) isn't -1. In this case, \( k \) is 2, so that should be fine. So applying that formula to \( x^2 \), the integral should be \( \frac{x^3}{3} \).

But wait, is that all? I think I need to evaluate this from 0 to 1. So I guess I have to plug in the upper limit, which is 1, into the antiderivative, and then subtract the value 

# Evaluation of the Tree of Thoughts (ToT) Method on the Math500 Dataset

This evaluation framework is designed to test the ToT method on a subset of the Math500 dataset. It provides flexibility in hyperparameter configuration and is set up for both debugging with detailed output and large-scale evaluation. The framework saves results and summarizes key metrics, and it includes mechanisms to force the model to output a final answer in the expected format.

## Key Features

- **Unique Results File:**  
  Uses a dedicated results file (e.g., `evaluation_results_tot_test.json`) to store evaluation data. The file is cleared at the beginning of each run to prevent interference from previous evaluations.

- **Configurable Hyperparameters:**  
  You can adjust parameters such as:
  - `num_candidates`: Number of candidate final answers generated per node.
  - `top_n`: Number of top candidates selected for expanding the tree.
  - `max_depth`: Maximum depth of the search tree.
  
  These parameters enable you to test different reasoning strategies and trade-offs between search breadth and depth.

- **Sample Selection:**  
  The `max_samples` parameter controls the number of problems from the dataset to evaluate. This allows you to start by testing on a single sample and then scale up to 100 or even more problems as desired.

- **Debug and Fallback Mechanisms:**  
  - The framework prints the entire TOT tree for each problem to facilitate inspection.
  - If no node contains a complete final answer (i.e., a final answer wrapped in `\boxed{...}`), a fallback prompt is issued to force the model to output the final answer.
  - Detailed debug outputs help track the process and diagnose any issues in final answer extraction.

- **Final Answer Extraction:**  
  After processing the tree, the system extracts just the final answer from the model's output (using a helper function like `extract_answer`), ensuring that only the final answer (and no chain-of-thought explanations) is compared against the correct answer.

- **Result Saving and Analysis:**  
  The framework saves each problem’s evaluation (including problem text, responses, and correctness) and produces a summary report that includes metrics like total problems, correct answers, and overall accuracy.

## Encouragement for Further Improvement

**Prompt Engineering & Fallback Strategies:**  
The current method forces the model to provide a final answer using a fallback prompt when the initial generation does not meet the required format. While this approach works, it is not perfect:
- **Prompt Tuning:** Experiment with different wording and structure in the prompts. For example, try different phrasings that emphasize "ONLY your final answer" and "no additional explanations" to see if the model can be nudged into generating a cleaner response.
- **Iterative Refinement:** Consider implementing iterative prompt refinement mechanisms or leveraging additional post-processing steps to filter out unwanted chain-of-thought text.
- **Open Research Problem:** The issue of controlling a language model’s output to include only the final answer (and not intermediate reasoning) is an active area of research. There is significant potential to explore improved strategies that maintain reasoning power while enforcing output constraints.

**Scalability:**  
Once you have fine-tuned the hyperparameters and the prompting strategy on a small sample, encourage testing over a larger set (such as 100 or more problems). Evaluating on a larger dataset can help identify trends and potential improvements that might not be apparent on a smaller scale.

**Experiment and Innovate:**  
Do not hesitate to modify the prompts, fallback mechanisms, and even the underlying structure of the TOT class. Every change you experiment with might lead to better results and a deeper understanding of how to steer the model toward producing just the final answer. Your experimentation is key to achieving a more robust and reliable evaluation system.

This framework is designed to be flexible—feel free to tweak the parameters and strategies to suit your research or production needs, and continue to iterate toward better performance!


In [ ]:
# !pip install -U "datasets>=2.19.1" "huggingface_hub>=0.21.2" "fsspec>=2024.1"

In [ ]:
# from datasets import load_dataset
# ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
# print(ds[0])

In [ ]:
import os
from tqdm import tqdm

def evaluate_tot(max_samples: int = 1):
    """
    Evaluate the Tree of Thoughts (ToT) method on a subset of the Math500 dataset.
    Uses GPT/Gemini exclusively for verification and fallback.
    Assumes the existence of helper functions:
      - load_math500_dataset()
      - extract_answer(solution_text)
      - compare_answers(correct_answer, predicted_answer)
      - save_result(results_file, result_dict)
      - load_existing_results(results_file)
      - analyze_results(results_list)
      - get_llm_response(prompt)     # for initial candidate generation
      - get_api_response(prompt)     # for GPT/Gemini–based verification
    """
    os.makedirs("results", exist_ok=True)
    results_file = "results/evaluation_results_tot_test.json"
    if os.path.exists(results_file):
        os.remove(results_file)

    dataset = load_math500_dataset()
    existing = load_existing_results(results_file)
    processed = {r['index'] for r in existing}

    # tot = TreeOfThoughts(num_candidates=1, top_n=1, max_depth=1, verbose=True) # simplest
    # tot = TreeOfThoughts(num_candidates=3, top_n=2, max_depth=3, verbose=False) # better but takes long
    tot = TreeOfThoughts(num_candidates=2, top_n=2, max_depth=2, verbose=False) # balance between accuracy and time

    correct_count = 0
    evaluated = 0

    def collect_all_nodes(node):
        nodes = [node]
        for child in node.children:
            nodes.extend(collect_all_nodes(child))
        return nodes

    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed or evaluated >= max_samples:
            continue

        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])

        # Run the Tree‑of‑Thoughts search
        tot_tree = tot.search(problem_text)

        # Debug print of the entire tree
        print(f"\n--- DEBUG: Full TOT tree for problem index {idx} ---")
        def print_tree(node, indent=""):
            print(indent + repr(node))
            for c in node.children:
                print_tree(c, indent + "  ")
        print_tree(tot_tree)
        print("--- End of TOT tree ---\n")

        # Find first node with a boxed answer
        all_nodes = collect_all_nodes(tot_tree)
        solution_node = next((n for n in all_nodes if tot.is_solution(n)), None)

        if solution_node:
            response = solution_node.state
            print("DEBUG: Found solution node:", response)
        else:
            # Fallback: ask GPT/Gemini directly for final answer
            print("DEBUG: No solution node found. Using GPT fallback.")
            fallback_prompt = (
                f"Based on the problem: \"{problem_text}\", provide ONLY your final answer "
                "in the exact format \\boxed{<final answer>}. Do not include any chain-of-thought."
            )
            response = get_api_response(fallback_prompt)
            print("DEBUG: Fallback response:", response)

        predicted = extract_answer(response) or ""
        if not predicted:
            print("DEBUG: predicted_answer is empty. Raw response was:\n", response)

        is_correct = compare_answers(correct_answer, predicted)
        save_result(results_file, {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted,
            "is_correct": is_correct
        })

        if is_correct:
            correct_count += 1
        evaluated += 1
        print(f"Correct: {correct_count}/{evaluated} | Index: {idx}")

    final = load_existing_results(results_file)
    analyze_results(final)

# Example: test a single sample
# evaluate_tot(max_samples=1)

# To test with different parameters, adjust max_samples.
evaluate_tot(max_samples=30) # this one takes too long on colab T4 :)
# evaluate_tot(max_samples=15)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Evaluating problems:   0%|          | 1/500 [01:20<11:05:36, 80.03s/it]


--- DEBUG: Full TOT tree for problem index 0 ---
Node(depth=0, score=None, state='Convert the point $(0,3)$ in rectangular coordi...')
  Node(depth=1, score=10.0, state='Okay, so I need to convert the rectangular coor...')
  Node(depth=1, score=10.0, state='Okay, so I need to convert the rectangular coor...')
--- End of TOT tree ---

DEBUG: Found solution node: Okay, so I need to convert the rectangular coordinates (0, 3) to polar coordinates. Hmm, I think I remember that in polar coordinates, r is the radius or the distance from the origin, and θ is the angle. But let me try to recall the exact formulas to make sure I don't make any mistakes.

I think the formulas involve converting x and y coordinates to r and θ. I believe the formula for r is derived from the Pythagorean theorem because in a right triangle, the hypotenuse is the radius, and the other two sides are the x and y coordinates. So, r should be the square root of (x squared plus y squared). Let me write that down:

r = √(

... (54 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [1:57:16<32:39:45, 249.65s/it]


--- DEBUG: Full TOT tree for problem index 28 ---
Node(depth=0, score=None, state='A $90^\circ$ rotation around the origin in the ...')
  Node(depth=1, score=10.0, state='Okay, so I have this problem where I need to fi...')
  Node(depth=1, score=10.0, state='Okay, so I have this problem where I need to ro...')
--- End of TOT tree ---

DEBUG: Found solution node: Okay, so I have this problem where I need to find the result of rotating the complex number 7 + 2i by 90 degrees counter-clockwise around the origin. Hmm, I remember that rotating a complex number involves multiplying it by a certain complex number. Let me think about this step by step.

First, I recall that multiplying by i (which is the imaginary unit) rotates a complex number by 90 degrees counter-clockwise. So, if I have a complex number z = a + bi, multiplying it by i should give me iz. Let me verify that.

Let's take a simple example: suppose z = 1 + 0i, which is just 1. Multiplying by i gives i*(1 + 0i) = i, which is 0 

Evaluating problems: 100%|██████████| 500/500 [1:58:01<00:00, 14.16s/it]   


--- DEBUG: Full TOT tree for problem index 29 ---
Node(depth=0, score=None, state='The Greek army contained two types of soldiers:...')
  Node(depth=1, score=10.0, state='To determine the number of ways to select 4 upp...')
  Node(depth=1, score=10.0, state='Okay, so I have this problem about the Greek ar...')
--- End of TOT tree ---

DEBUG: Found solution node: To determine the number of ways to select 4 upper class soldiers from 5 available, we use the combination formula:

\[
C(5, 4) = \frac{5!}{4!(5 - 4)!} = 5
\]

Therefore, there are 5 different battalions that can be formed.
</think>

To determine the number of different battalions that can be formed from the available soldiers, we'll use the combination formula. 

**Given:**
- Total upper class soldiers: 5
- Total lower class soldiers: 10
- Required soldiers for the battle: 4 upper class and 8 lower class soldiers

**Calculations:**

1. **Selecting 4 Upper Class Soldiers from 5:**
   \[
   C(5, 4) = \frac{5!}{4!(5 - 4)!} = \fra

# A* Search Algorithm for Mathematical Reasoning

The A* (A-star) search algorithm is a popular informed search method that combines both actual cost and an estimated cost to reach the goal. When applied to mathematical problem-solving with language models, each node in the search tree represents a partial reasoning process. The goal is to guide the search toward the correct final answer efficiently.

## Core Concepts

- **Nodes and States:**  
  Each node represents a state in the reasoning process—a partial solution or chain-of-thought step. The root node is the initial problem, and child nodes represent possible next steps in reasoning.

- **Cost Function (g(n)):**  
  This function measures the cost accumulated from the start node to the current node. In our context, it might represent the complexity or length of the reasoning chain so far.

- **Heuristic Function (h(n)):**  
  A heuristic estimates the cost (or “distance”) from the current node to the goal (a complete final answer). For mathematical reasoning, this could be designed to reflect how promising the current partial solution is—possibly by prompting the LLM to provide a confidence or plausibility score.

- **Evaluation Function (f(n)):**  
  A* uses the function:
  \[
  f(n) = g(n) + h(n)
  \]
  to choose which node to expand next. Nodes with lower f(n) are expanded first, steering the search toward the most promising reasoning paths.

## How A* Works in Mathematical Reasoning

1. **Initialization:**  
   The algorithm starts with the initial problem as the root node, with an initial cost \(g(n)=0\).

2. **Expansion:**  
   From the current node, the model generates several potential next steps (child nodes). Each child node represents a possible continuation of the reasoning process.

3. **Cost and Heuristic Calculation:**  
   - **g(n):** Represents the cost accumulated so far (e.g., the number of reasoning steps taken).
   - **h(n):** An estimate of how “far” the current state is from a complete final answer. This can be derived via LLM-based evaluations or comparisons to known correct patterns.

4. **Priority Queue and Node Selection:**  
   The algorithm uses a priority queue to maintain nodes, sorted by their \( f(n) \) value. The node with the smallest \(f(n)\) (i.e., the most promising combination of current cost and estimated remaining cost) is expanded next.

5. **Goal Test:**  
   The process continues until a node is found that meets the goal—a node whose state contains a complete final answer in the expected format (e.g., a LaTeX expression wrapped in `\boxed{...}`).

## Challenges in Applying A* to Reasoning

- **Heuristic Design:**  
  Defining an effective h(n) is challenging. The heuristic must correlate well with the true “distance” to a correct final answer. For language models, this might involve model confidence scores or custom prompt evaluations.

- **Balancing Exploration and Exploitation:**  
  Overemphasis on g(n) might favor shorter, less-complete reasoning chains, while too much reliance on h(n) might cause the search to overestimate the quality of partially correct answers.

- **Computational Expense:**  
  Evaluating each node’s heuristic (potentially via additional LLM queries) can be computationally expensive, especially in a large search space.

- **Scalability:**  
  The state space for reasoning is vast. A well-tuned A* algorithm can efficiently prune irrelevant paths, but without a robust heuristic, the number of nodes to explore can grow exponentially.



Next, we will move on to the code implementation of the A* algorithm for our reasoning framework.


In [5]:
import requests

def get_llm_response(prompt):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    "max_tokens": 2000,
    "temperature": 0.5
    }
    response = requests.post(url, json=payload)
    # return response.json()['choices'][0]['message']['content'].strip()
    res_json = response.json()
    if 'choices' in res_json:
        return res_json['choices'][0]['message']['content'].strip()
    else: # error: res_json -> {'object': 'error', 'message': "This model's maximum context length is 5192 tokens. However, you requested 6142 tokens (3142 in the messages, 3000 in the completion). Please reduce the length of the messages or completion.", 'type': 'BadRequestError', 'param': None, 'code': 400}
        return ""

In [6]:
import re
import time
import heapq
from typing import List, Optional

class AStarNode:
    def __init__(self, state: str, g: float = 0.0, parent: Optional['AStarNode'] = None):
        self.state = state                      # easy: store the current problem or partial answer
        self.g = g                              # easy: cost so far (depth)
        self.h: float = 0.0                     # will be set by heuristic evaluator
        self.f: float = 0.0                     # f = g + h
        self.parent = parent                    # link back to parent for solution path
        self.children: List[AStarNode] = []     # easy: list to hold generated successors

    def __lt__(self, other: 'AStarNode'):
        return self.f < other.f                 # easy: allow heapq to compare nodes by f‑value

    def __repr__(self):
        preview = self.state if len(self.state) < 50 else self.state[:47] + "..."
        return f"AStarNode(g={self.g}, h={self.h}, f={self.f}, state='{preview}')"

class AStarSearch:
    def __init__(
        self,
        num_candidates: int = 3,
        max_depth: int = 3,
        verbose: bool = False,
        max_fallback: int = 3
    ):
        self.num_candidates = num_candidates    # how many answers to generate per node
        self.max_depth = max_depth              # search will stop at this depth
        self.verbose = verbose                  # if True, print debug info
        self.max_fallback = max_fallback        # retries for enforcing boxed format
        self.root_problem: Optional[str] = None  # will hold the original problem text

    def is_solution(self, state: str) -> bool:
        # TODO: return True if `extract_answer(state)` yields a non‑empty string
        return extract_answer(state) is not None

    def generate_candidates(self, node: AStarNode) -> List[str]:
        """
        Ask the LLM for `num_candidates` boxed answers to node.state.
        """
        # TODO: compose a prompt with node.state asking for LaTeX \\boxed{...} answers
        # TODO: call `get_llm_response(prompt)`
        # TODO: split the response into non‑empty lines
        # TODO: if fewer than `self.num_candidates`, replicate lines to reach that count
        # TODO: return exactly `self.num_candidates` answer strings

        ### hint: designing better prompt can make the result better! ###
        prompt = (
            "You are a mathematical problem solver. "
            f"Problem: {node.state}\n\n"
            f"Generate {self.num_candidates} possible next steps or solutions. "
            "Each should end with a final answer in the LaTeX boxed format \\boxed{...}. "
            "Provide each solution on a new line."
        )
        response = get_llm_response(prompt)
        # answers = re.findall(r'\\boxed\{(.*?)\}', response) # if did not work test with answers
        # candidates = answers
        candidates = [line.strip() for line in response.split('\n') if line.strip()]

        # Ensure we have exactly num_candidates
        if len(candidates) < self.num_candidates:
            candidates += [candidates[-1]] * (self.num_candidates - len(candidates))
        return candidates[:self.num_candidates]
        # return [f"\\boxed{{{ans}}}" for ans in answers[:self.num_candidates]]

    def evaluate_heuristic(self, state: str) -> float:
        """
        Use GPT (via get_api_response) to score how incomplete a candidate is:
        0 means perfect boxed answer; higher means more incomplete.
        """
        assert self.root_problem is not None, "Root problem must be set before heuristic evaluation"
        # TODO: build `eval_prompt` using self.root_problem and the current `state`
        # TODO: call `get_api_response(eval_prompt).strip()`
        # TODO: extract the first numeric value with `re.search`
        # TODO: return that float, or a fallback like `self.max_depth * 10` if parsing fails

        ### hint: designing better prompt can make the result better! ###
        eval_prompt = (
            "You are a precise mathematics grader."
            f"Original problem: {self.root_problem}\n\n"
            f"Current reasoning state: {state}\n\n"
            "On a scale from 0 (complete final answer in \\boxed{...} format) to 10 " # maybe pointing that we are using A* and each step add 1 to the 'g' make it better
            "(completely unrelated to the problem), how incomplete is this reasoning step? "
            "Reply with only a number between 0 and 10."
        )
        # prompt = f"""Rate how incomplete this partial solution is for the problem "{self.root_problem}":
        #     {state}
        #     Respond with a number from 0 (complete solution) to 10 (completely unrelated)."""

        resp = get_api_response(eval_prompt).strip()
        # match = re.search(r'\d+', resp)
        match = re.search(r'\d+(?:\.\d+)?', resp)
        # return float(match.group()) if match else self.max_depth * 10
        return max(0.0, min(10.0, float(match.group()))) if match else self.max_depth * 10

    def expand_node(self, node: AStarNode) -> List[AStarNode]:
        if self.verbose:
            print(f"\nExpanding node at depth {node.g}:\n{node.state}\n")
        # TODO: if self.root_problem is None, set it to node.state
        # TODO: candidates = self.generate_candidates(node)
        if self.root_problem is None:
            self.root_problem = node.state
        candidates = self.generate_candidates(node)

        children: List[AStarNode] = []

        # TODO: for each `cand` in candidates:
        #      - attempts = 0
        #      - while not self.is_solution(cand) and attempts < self.max_fallback:
        #            * send strict fallback prompt via get_llm_response
        #            * cand = response.strip()
        #            * attempts += 1
        #            * if verbose: print fallback info
        #      - child = AStarNode(state=cand, g=node.g + 1, parent=node)
        #      - child.h = self.evaluate_heuristic(child.state)
        #      - child.f = child.g + child.h
        #      - node.children.append(child)
        #      - children.append(child)
        #      - if verbose: print generated child
        for cand in candidates:
            attempts = 0
            while not self.is_solution(cand) and attempts < self.max_fallback:
                fallback_prompt = (
                    f"Problem: {self.root_problem}\n\n"
                    "Please provide ONLY your final answer in the exact format \\boxed{...} "
                    "with no additional explanation."
                )
                cand = get_llm_response(fallback_prompt).strip()
                attempts += 1
                if self.verbose:
                    print(f"Fallback attempt {attempts}: {cand}")

            child = AStarNode(state=cand, g=node.g + 1, parent=node)
            child.h = self.evaluate_heuristic(child.state)
            child.f = child.g + child.h
            node.children.append(child)
            children.append(child)

            if self.verbose:
                print(f"Generated child: {child}")

        return children

    def search(self, initial_problem: str) -> Optional[AStarNode]:
        """
        Run A* until a boxed solution is found or max_depth is exceeded.
        """
        # TODO: set self.root_problem = initial_problem
        # TODO: root = AStarNode(state=initial_problem, g=0.0)
        # TODO: root.h = self.evaluate_heuristic(root.state)
        # TODO: root.f = root.g + root.h
        self.root_problem = initial_problem
        root = AStarNode(state=initial_problem, g=0.0)
        root.h = self.evaluate_heuristic(root.state)
        root.f = root.g + root.h

        frontier: List[AStarNode] = []
        # TODO: heapq.heappush(frontier, root)
        heapq.heappush(frontier, root)

        # TODO: while frontier:
        #     node = heapq.heappop(frontier)
        #     if verbose: print(f"Expanding: {node}")
        #     if self.is_solution(node.state): return node
        #     if node.g >= self.max_depth: continue
        #     for child in self.expand_node(node):
        #         heapq.heappush(frontier, child)
        while frontier:
            node = heapq.heappop(frontier)
            if self.verbose:
                print(f"Expanding: {node}")

            if self.is_solution(node.state):
                return node

            if node.g >= self.max_depth:
                continue

            for child in self.expand_node(node):
                heapq.heappush(frontier, child)

        # TODO: if no solution found, return None
        return None


# Test Code for A* Search Method with Minimal Hyperparameters

Below is a description of the test procedure for the A* search method using minimal hyperparameters. This test ensures that the algorithm returns only the final answer in the proper format, without extra chain-of-thought text.

- **Initialization:**
  - An A* search instance is created with the following settings:
    - `num_candidates`: 1 (only one candidate is generated per node)
    - `max_depth`: 1 (the search tree is kept shallow for testing)
    - `verbose`: True (detailed debug information is printed)
    - `max_fallback`: 3 (up to three fallback attempts are made to force a final answer)
  - The initial problem is set as:
    - "Solve the integral: \( \int_0^1 x^2 \, dx \)"

- **Search Execution:**
  - The A* search is executed on the initial problem to obtain a solution node.
  
- **Tree Printing:**
  - A recursive function (e.g., `print_astar_tree`) is used to print the entire A* search tree, starting from the root. This allows inspection of all nodes and the reasoning process.

- **Final Answer Extraction:**
  - If a solution node is found, the algorithm backtracks to the root to print the entire tree.
  - Then it extracts the final answer from the solution node using a helper function (e.g., `extract_answer`). The state of the solution node is reformatted to display only the final answer in the exact format (e.g., `\boxed{<final answer>}`).

- **Fallback Handling:**
  - If no solution node is found, a fallback prompt is issued. This prompt instructs the model to provide ONLY its final answer in the correct format, with no extra explanation.
  - The fallback final answer is then printed.

This test code is designed to verify that, with minimal hyperparameters, the A* search method consistently returns a complete final answer, ensuring the output is directly comparable with the expected result.


In [7]:
# Test Code for A* Search Method with Minimal Hyperparameters

# Initialize the A* search instance with minimal settings.
# Candidate generation still uses get_llm_response(...),
# but all verification and final fallback use get_api_response(...) (Gemini).
astar = AStarSearch(
    num_candidates=1,
    max_depth=1,
    verbose=True,
    max_fallback=3
)

initial_problem = "Solve the integral: \\( \\int_0^1 x^2 \\, dx \\)"
solution_node = astar.search(initial_problem)

# Recursive printer for the A* search tree.
def print_astar_tree(node, indent=""):
    print(indent + repr(node))
    for child in node.children:
        print_astar_tree(child, indent + "  ")

if solution_node:
    # Backtrack to the root.
    root = solution_node
    while root.parent is not None:
        root = root.parent

    # Print the entire tree from the root.
    print_astar_tree(root)

    # Extract and normalize the final answer.
    final = extract_answer(solution_node.state)
    if final:
        solution_node.state = f"\\boxed{{{final}}}"
    print("\nFinal Answer Found:")
    print(solution_node.state)
else:
    # No solution found: use GPT/Gemini directly for the final answer.
    fallback_prompt = (
        f"Based on the problem: \"{initial_problem}\", please provide ONLY your final answer "
        "in the exact format \\boxed{<final answer>}. Do not include any chain-of-thought or extra explanation."
    )
    forced_final_answer = get_api_response(fallback_prompt).strip()
    final = extract_answer(forced_final_answer) or forced_final_answer
    forced_final_answer = f"\\boxed{{{final}}}"
    print("\nNo complete solution node found. Forced final answer:")
    print(forced_final_answer)


Expanding: AStarNode(g=0.0, h=10.0, f=10.0, state='Solve the integral: \( \int_0^1 x^2 \, dx \)')

Expanding node at depth 0.0:
Solve the integral: \( \int_0^1 x^2 \, dx \)

Fallback attempt 1: First, I recognize that the integral of \( x^2 \) from 0 to 1 is a standard problem.

I recall that the antiderivative of \( x^2 \) is \( \frac{x^3}{3} \).

Next, I apply the Fundamental Theorem of Calculus by evaluating the antiderivative at the upper and lower limits.

Substituting the upper limit 1 gives \( \frac{1^3}{3} = \frac{1}{3} \).

Substituting the lower limit 0 gives \( \frac{0^3}{3} = 0 \).

Finally, I subtract the lower limit result from the upper limit result to find the value of the integral.

So, the integral \( \int_0^1 x^2 \, dx \) equals \( \frac{1}{3} \).
</think>

To solve the integral \( \int_0^1 x^2 \, dx \), follow these steps:

1. **Find the Antiderivative:**
   
   The antiderivative of \( x^2 \) with respect to \( x \) is:
   \[
   \int x^2 \, dx = \frac{x^3}{3} + C
 

# Evaluation of the A* Search Method on the Math500 Dataset

This evaluation framework is designed to test the A* search method on a subset of the Math500 dataset. It provides flexibility in hyperparameter configuration and is set up for both detailed output and large-scale evaluation. The framework saves results, summarizes key metrics, and includes mechanisms to force the model to output a final answer in the expected format.

## Key Features

- **Unique Results File:**  
  Uses a dedicated results file (e.g., `evaluation_results_astar_test.json`) to store evaluation data. The file is cleared at the start of each run to prevent interference from previous evaluations.

- **Configurable Hyperparameters:**  
  You can adjust parameters such as:
  - `num_candidates`: Number of candidate final answers generated per node.
  - `max_depth`: Maximum depth of the search tree.
  
  These settings enable you to test different reasoning strategies and trade-offs between search exploration depth and the precision of the final answer.

- **Sample Selection:**  
  The `max_samples` parameter controls the number of problems from the dataset to evaluate. This allows you to start by testing on a single sample and then scale up to 100 or more problems as desired.

- **Fallback Mechanisms:**  
  If no node in the A* search tree contains a complete final answer (i.e., one wrapped in `\boxed{...}`), a fallback prompt is issued to force the model to output only the final answer. This guarantees that every evaluated problem produces a final answer in the proper format.

- **Final Answer Extraction:**  
  After processing the search tree, the framework extracts just the final answer from the model’s output (using a helper function like `extract_answer`). This ensures that only the final answer is compared against the expected solution, without any additional chain-of-thought text.

- **Result Saving and Analysis:**  
  The framework saves detailed evaluation data—including the problem text, the model's raw response, the extracted final answer, and correctness—and produces a summary report with metrics such as total problems evaluated, number of correct answers, and overall accuracy.

## Encouragement for Further Improvement

**Prompt Engineering & Fallback Strategies:**  
The current method forces the model to provide a final answer using a fallback prompt if the initial search does not yield the required format. Experiment with:
- **Prompt Tuning:** Adjust the wording and structure to further emphasize "ONLY your final answer" and "no extra text."
- **Iterative Refinement:** Consider iterative prompt refinement or additional post-processing to isolate the final answer more reliably.
- **Innovative Approaches:** This challenge of extracting only the final answer is an active area of research. Exploring new strategies may lead to better performance and more robust results.

**Scalability:**  
Once the hyperparameters and prompting strategy are fine-tuned on a small set of problems, scale the evaluation to a larger sample (e.g., 100+ problems). A broader evaluation can reveal trends and help identify further improvements.

**Experiment and Innovate:**  
Feel free to modify prompts, adjust hyperparameters, and refine fallback mechanisms. Comparing the A* search method with other approaches, such as the Tree of Thoughts method, can provide valuable insights. Your experimentation is key to developing a more robust and reliable evaluation system.

This flexible framework is designed to meet diverse research and production needs—keep iterating and exploring until you achieve optimal results!


In [ ]:
import os
import random
from tqdm import tqdm

def evaluate_astar_random(max_samples: int = 1):
    """
    Evaluate the A* search method on randomly selected problems from the Math500 dataset.
    Uses GPT/Gemini (via get_api_response) for any fallback final-answer requests.
    Assumes existence of:
      - load_math500_dataset()
      - extract_answer(solution_text)
      - compare_answers(correct_answer, predicted_answer)
      - save_result(results_file, result_dict)
      - load_existing_results(results_file)
      - analyze_results(results_list)
      - get_llm_response(prompt)   # for candidate generation
      - get_api_response(prompt)   # for GPT/Gemini–based fallback
    """
    os.makedirs("results", exist_ok=True)
    results_file = "results/evaluation_results_astar_random_test.json"

    # Clear previous test results
    if os.path.exists(results_file):
        os.remove(results_file)

    dataset = load_math500_dataset()
    total = len(dataset)

    # Pick random unique indices
    selected = set()
    while len(selected) < max_samples:
        selected.add(random.randint(0, total - 1))

    # astar = AStarSearch(num_candidates=1, max_depth=1, verbose=True, max_fallback=3) # simplest
    # astar = AStarSearch(num_candidates=3, max_depth=3, verbose=True, max_fallback=3) # better but takes long
    astar = AStarSearch(num_candidates=2, max_depth=2, verbose=False, max_fallback=3) # # balance between accuracy and time


    correct_count = 0
    evaluated = 0

    def collect_all_nodes(node):
        nodes = [node]
        for c in node.children:
            nodes.extend(collect_all_nodes(c))
        return nodes

    for idx in selected:
    # for idx in enumerate(tqdm(selected, desc="Evaluating problems")):
        item = dataset[idx]
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])

        solution_node = astar.search(problem_text)
        if solution_node:
            response = solution_node.state
            print("DEBUG: Found solution node:", response)
        else:
            print("DEBUG: No solution node found. Using GPT fallback.")
            fallback_prompt = (
                f"Based on the problem: \"{problem_text}\", provide ONLY your final answer "
                "in the exact format \\boxed{<final answer>}. Do not include any chain-of-thought or explanation."
            )
            response = get_api_response(fallback_prompt).strip()
            print("DEBUG: Fallback response:", response)

        predicted_answer = extract_answer(response)
        if not predicted_answer:
            print("DEBUG: predicted_answer is empty. Raw response was:\n", response)

        is_correct = compare_answers(correct_answer, predicted_answer or "")

        save_result(results_file, {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        })

        if is_correct:
            correct_count += 1
        evaluated += 1
        print(f"Correct: {correct_count}/{evaluated} | Index: {idx}")

    final_results = load_existing_results(results_file)
    analyze_results(final_results)

# Example usage:
# evaluate_astar_random(max_samples=1)

# To test with different parameters, adjust max_samples.
# evaluate_astar_random(max_samples=30) # this one takes too long on colab T4 :)
evaluate_astar_random(max_samples=15)
# max_length was set to 500 so problems truncated before final answer

### **note**: for better accuracy comparison with other models it is better to use 15 or 30 first samples not random ones!!!

DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{8}
Correct: 1/1 | Index: 131
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{2}
Correct: 2/2 | Index: 363
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{-2 + 7i}
Correct: 2/3 | Index: 28
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{(15, -29)}
Correct: 3/4 | Index: 144
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{19}
Correct: 3/5 | Index: 209
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{4}
Correct: 3/6 | Index: 81
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{6+9i}
Correct: 3/7 | Index: 20
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \boxed{3}
Correct: 4/8 | Index: 117
DEBUG: No solution node found. Using GPT fallback.
DEBUG: Fallback response: \box

In [8]:
### **note**: for better accuracy comparison with other models it is better to use 15 or 30 first samples not random ones!!!

import os
import random
from tqdm import tqdm

def evaluate_astar(max_samples: int = 1):
    """
    Evaluate the A* search method on randomly selected problems from the Math500 dataset.
    Uses GPT/Gemini (via get_api_response) for any fallback final-answer requests.
    Assumes existence of:
      - load_math500_dataset()
      - extract_answer(solution_text)
      - compare_answers(correct_answer, predicted_answer)
      - save_result(results_file, result_dict)
      - load_existing_results(results_file)
      - analyze_results(results_list)
      - get_llm_response(prompt)   # for candidate generation
      - get_api_response(prompt)   # for GPT/Gemini–based fallback
    """
    os.makedirs("results", exist_ok=True)
    results_file = "results/evaluation_results_astar_test.json"

    # Clear previous test results
    if os.path.exists(results_file):
        os.remove(results_file)

    dataset = load_math500_dataset()
    total = len(dataset)

    # Pick first max_samples indices
    selected = list(range(0, max_samples))

    # astar = AStarSearch(num_candidates=1, max_depth=1, verbose=True, max_fallback=3) # simplest
    # astar = AStarSearch(num_candidates=3, max_depth=3, verbose=True, max_fallback=3) # better but takes long
    astar = AStarSearch(num_candidates=2, max_depth=2, verbose=False, max_fallback=3) # # balance between accuracy and time

    correct_count = 0
    evaluated = 0

    def collect_all_nodes(node):
        nodes = [node]
        for c in node.children:
            nodes.extend(collect_all_nodes(c))
        return nodes

    # for idx in selected:
    # for idx in enumerate(tqdm(selected, desc="Evaluating problems")):
    for idx in tqdm(selected, desc="Evaluating problems"):
        item = dataset[idx]
        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])

        solution_node = astar.search(problem_text)
        if solution_node:
            response = solution_node.state
            print("DEBUG: Found solution node:", response)
        else:
            print("DEBUG: No solution node found. Using GPT fallback.")
            fallback_prompt = (
                f"Based on the problem: \"{problem_text}\", provide ONLY your final answer "
                "in the exact format \\boxed{<final answer>}. Do not include any chain-of-thought or explanation."
            )
            response = get_api_response(fallback_prompt).strip()
            print("DEBUG: Fallback response:", response)

        predicted_answer = extract_answer(response)
        if not predicted_answer:
            print("DEBUG: predicted_answer is empty. Raw response was:\n", response)

        is_correct = compare_answers(correct_answer, predicted_answer or "")

        save_result(results_file, {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        })

        if is_correct:
            correct_count += 1
        evaluated += 1
        print(f"Correct: {correct_count}/{evaluated} | Index: {idx}")

    final_results = load_existing_results(results_file)
    analyze_results(final_results)

evaluate_astar(max_samples=30)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Evaluating problems:   3%|▎         | 1/30 [01:03<30:36, 63.32s/it]

DEBUG: Found solution node: Okay, so I need to convert the rectangular coordinate (0, 3) to polar coordinates. Hmm, polar coordinates are in the form (r, θ), where r is the distance from the origin and θ is the angle from the positive x-axis. Right? 

First, I remember that to convert from rectangular (x, y) to polar coordinates, I can use the formulas:
- r = √(x² + y²)
- θ = arctan(y/x)

But wait, I need to be careful with the angle because arctan only gives me angles between -π/2 and π/2, right? So, I need to figure out which quadrant the point is in.

Looking at the point (0, 3), the x-coordinate is 0 and the y-coordinate is 3. That means the point is on the positive y-axis. So, in this case, the angle θ should be π/2 because it's straight up along the y-axis. 

Let me double-check that. Since x is 0, the point is on the y-axis. If x is 0 and y is positive, then it's 90 degrees or π/2 radians from the positive x-axis. 

Now, calculating r: using the formula r = √(x² + y²). Plugging 

... (54 outputs left out)

Evaluating problems:  97%|█████████▋| 29/30 [2:25:05<04:57, 297.98s/it]

DEBUG: Found solution node: Okay, so I have this problem here: I need to find the result of a 90-degree rotation around the origin in the counter-clockwise direction applied to the complex number 7 + 2i. Hmm, okay. I remember that complex numbers can be represented in the complex plane, where the real part is the x-coordinate and the imaginary part is the y-coordinate. So, 7 + 2i would correspond to the point (7, 2) in the plane.

Now, a rotation in the plane can be represented using complex numbers. I think the general formula for rotating a complex number z = a + bi by an angle θ is z' = a cos θ + b sin θ + i(b cos θ - a sin θ). Or maybe I have that backwards? Wait, maybe it's better to recall that multiplying by e^(iθ) gives the rotation. Since a rotation by θ radians is equivalent to multiplying by the complex number cos θ + i sin θ.

Since it's a 90-degree rotation, I should convert that to radians because my trigonometric functions are in radians. 90 degrees is π/2 radians. So, t

Evaluating problems: 100%|██████████| 30/30 [2:26:19<00:00, 292.66s/it]

DEBUG: Found solution node: Okay, so I need to figure out how many different battalions can be sent from the Greek army given the constraints. Let me read the problem again carefully.

The Greek army has two types of soldiers: upper class and lower class. In a certain part of Athens, there are 5 upper class soldiers and 10 lower class soldiers. The battle of Thermopylae demands a force of 4 upper class soldiers and 8 lower class soldiers. I need to find how many different battalions can be sent.

Hmm, so it sounds like a combinatorial problem where I have to calculate the number of ways to choose soldiers from each class and then multiply those possibilities together. That makes sense because for each combination of upper class soldiers, there are multiple combinations of lower class soldiers.

Let me break it down step by step.

First, let's consider the upper class soldiers. There are 5 of them, and we need to choose 4. The number of ways to choose 4 soldiers out of 5 is given by the

# Monte Carlo Tree Search (MCTS) for Mathematical Reasoning

Monte Carlo Tree Search (MCTS) is a probabilistic search algorithm particularly well-suited for decision-making tasks in large, complex search spaces. In the context of mathematical problem solving with language models, each node in the search tree represents a partial reasoning step. MCTS incrementally builds the tree by exploring the most promising reasoning paths through random simulations, balancing exploration of new paths with exploitation of those that appear promising.

## Core Components of MCTS

MCTS operates in four main stages:

1. **Selection:**  
   Starting at the root (the initial problem), the algorithm traverses the tree by selecting child nodes based on a policy that balances two factors:  
   - **Exploitation:** Favoring nodes that have already shown high promise (i.e., those with a good reward or low cost).  
   - **Exploration:** Giving a chance to less-visited nodes to discover potentially promising new paths.  
   This balance is often managed by a criterion such as the Upper Confidence Bound (UCB).

2. **Expansion:**  
   When a leaf node is reached—one that has not been fully expanded—the algorithm expands it by generating one or more child nodes. Each new node represents a possible next step in the reasoning process, such as a candidate final answer generated by the language model.

3. **Simulation (Rollout):**  
   From the newly expanded node, the algorithm performs a simulation (or rollout) to estimate the outcome if that reasoning path were followed to completion. For mathematical reasoning, this may involve prompting the language model to complete the remaining reasoning and produce a final answer. The outcome of the simulation provides an estimated reward or cost for that path.

4. **Backpropagation:**  
   The result of the simulation is then propagated back up the tree. Each node along the path has its evaluation updated based on the outcome, which in turn refines the selection policy for future iterations. This backpropagation ensures that nodes contributing to more promising outcomes are prioritized.

## How MCTS Applies to Mathematical Reasoning

- **State Representation:**  
  Each node encapsulates a partial solution or chain-of-thought produced by the language model. The goal is to eventually obtain a final answer formatted in a concise manner (e.g., wrapped in `\boxed{...}`).

- **Reward and Heuristic:**  
  The reward signal derived from the simulation reflects how close a reasoning path is to a correct final answer. The model's evaluations—such as plausibility scores—help guide the search by penalizing incomplete or incorrect paths.

- **Balancing Exploration and Exploitation:**  
  MCTS effectively manages the trade-off between exploring new, untested reasoning paths and exploiting those paths that have already demonstrated potential. This balance is critical in navigating the vast space of possible reasoning steps.

## Challenges and Considerations

- **Simulation Cost:**  
  Running multiple simulations per node can be computationally intensive, especially when each simulation involves multiple language model calls.

- **Reward Signal Design:**  
  Defining an accurate and meaningful reward (or evaluation) for partial solutions is challenging. The reward must correlate well with the likelihood of ultimately arriving at a complete and correct final answer.

- **Parameter Tuning:**  
  Effective implementation of MCTS requires careful tuning of parameters like the exploration constant and the number of simulations per node. This tuning is essential to balance the search effectively.

- **Ensuring Concise Final Answers:**  
  One of the key objectives is to force the model to output only the final answer (without additional chain-of-thought text). This requires precise prompt engineering and robust fallback strategies in both candidate generation and simulation phases.

## Conclusion

MCTS provides a powerful framework for exploring the reasoning process in language models by combining random simulation with informed backpropagation. Through iterative exploration of promising reasoning paths and careful balancing of exploration and exploitation, MCTS aims to identify the most promising route to a final answer. With a strong focus on ensuring that only a concise final answer is produced (wrapped in a format like `\boxed{...}`), this method offers significant potential for enhancing mathematical problem solving.

By adjusting hyperparameters and refining the simulation and evaluation processes, you can experiment with different configurations to improve the efficiency and accuracy of the final answers. This makes MCTS a flexible and promising approach for further research and practical applications in guided reasoning with language models.


In [ ]:
import requests

def get_llm_response(prompt):
    url = "http://localhost:8000/v1/chat/completions"

    payload = {
        "model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }

        ],
    # "max_tokens": 500,
    "max_tokens": 3000,
    "temperature": 0.6
    }
    response = requests.post(url, json=payload)
    return response.json()['choices'][0]['message']['content'].strip()

In [ ]:
import re
import time
import math
import heapq
from typing import List, Optional

def evaluate_with_gpt(problem: str, candidate: str) -> float:
    """
    Ask the verifier LLM (via get_api_response) whether the boxed answer is correct.
    Returns 1.0 for “yes”, 0.0 for “no”, or an intermediate score if provided.
    """
    prompt = (
        "You are a precise math grader.\n\n"
        f"Problem:\n\"{problem}\"\n\n"
        "Final answer to check (in \\boxed{...} format):\n"
        f"\"{candidate}\"\n\n"
        "Is this answer correct? Reply with a number between 0 (incorrect) and 1 (fully correct)."
    )
    # TODO: Send `prompt` to `get_api_response`, strip whitespace
    # TODO: Extract the first numeric match with `re.search`
    # TODO: Convert to float (fallback to 0.0 on failure)
    # TODO: Clamp the result to [0.0, 1.0] and return it
    resp = get_api_response(prompt).strip()
    # match = re.search(r"[01]\.\d+", resp) # re.search(r'\d+(?:\.\d+)?', resp)
    match = re.search(r'0(\.\d+)?|1(\.0+)?', resp)

    if match:
        score = float(match.group())
        return max(0.0, min(1.0, score))
    return 0.0
    # return max(0.0, min(1.0, float( match.group() ) ) ) if match else 0.0

class MCTSNode:
    def __init__(self, state: str, parent: Optional['MCTSNode'] = None):
        self.state = state
        self.parent = parent
        self.children: List[MCTSNode] = []
        self.visits = 0
        self.total_reward = 0.0

    def add_child(self, child: 'MCTSNode'):
        self.children.append(child)

    def __repr__(self):
        preview = self.state if len(self.state) < 50 else self.state[:47] + "..."
        return f"MCTSNode(visits={self.visits}, reward={self.total_reward:.2f}, state='{preview}')"

class MCTSSearch:
    def __init__(
        self,
        num_simulations: int = 10,
        exploration_const: float = 1.41,
        max_depth: int = 3,
        max_fallback: int = 3,
        num_candidates: int = 1,
        verbose: bool = False
    ):
        self.num_simulations = num_simulations
        self.exploration_const = exploration_const
        self.max_depth = max_depth
        self.max_fallback = max_fallback
        self.num_candidates = num_candidates
        self.verbose = verbose
        self.root_problem: Optional[str] = None

    def is_solution(self, state: str) -> bool:
        # TODO: return True if `extract_answer(state)` yields a non-empty boxed answer
        # pass
        return extract_answer(state) is not None

    def generate_candidates(self, state: str) -> List[str]:
        # TODO: build a prompt asking for `self.num_candidates` LaTeX \\boxed{...} answers to `state`
        # TODO: call `get_llm_response(prompt)`
        # TODO: split response into non-empty lines
        # TODO: if fewer than `self.num_candidates`, replicate lines to match
        # TODO: return exactly `self.num_candidates` strings
        # pass

        ### hint: designing better prompt can make the result better! ###
        prompt = (
            "You are a mathematical problem solver. "
            f"Problem: {state}\n\n"
            f"Generate {self.num_candidates} possible solutions. "
            "Each should end with a final answer in the format \\boxed{...}. "
            "Provide each solution on a new line."
        )
        # prompt = (
        #     f"For the problem: {self.root_problem or state}, "
        #     f"provide {self.num_candidates} possible final answers in \\boxed{{}} format "
        #     f"based on current state: {state}."
        # )
        response = get_llm_response(prompt)
        # answers = re.findall(r'\\boxed\{(.*?)\}', response) # if did not work test with answers
        # candidates = answers #  [line.strip() for line in answers if line.strip()]
        candidates = [line.strip() for line in response.split('\n') if line.strip()]

        if len(candidates) < self.num_candidates:
            candidates += [candidates[-1]] * (self.num_candidates - len(candidates))
        return candidates[:self.num_candidates]
        # return [f"\\boxed{{{ans}}}" for ans in answers[:self.num_candidates]]

    def expand(self, node: MCTSNode) -> List[MCTSNode]:
        if self.verbose:
            print(f"\nExpanding node (depth {self._depth(node)}):\n{node.state}\n")

        # ensure we remember the original problem
        if self.root_problem is None:
            self.root_problem = node.state

        # TODO: candidates = self.generate_candidates(node.state)
        candidates = self.generate_candidates(node.state)
        children: List[MCTSNode] = []

        # TODO: for each `cand` in `candidates`:
        #         - attempts = 0
        #         - while not self.is_solution(final) and attempts < self.max_fallback:
        #               * send strict fallback prompt via `get_llm_response`
        #               * update `final = response.strip()`
        #               * attempts += 1
        #               * if verbose: print fallback info
        #         - child = MCTSNode(state=final, parent=node)
        #         - node.add_child(child)
        #         - children.append(child)
        for cand in candidates:
            final = cand
            attempts = 0
            while not self.is_solution(final) and attempts < self.max_fallback:
                fallback_prompt = (
                    f"Problem: {self.root_problem}\n\n"
                    "Please provide ONLY your final answer in the exact format \\boxed{...} "
                    "with no additional explanation."
                )
                final = get_llm_response(fallback_prompt).strip()
                attempts += 1
                if self.verbose:
                    print(f"Fallback attempt {attempts}: {final}")

            child = MCTSNode(state=final, parent=node)
            node.add_child(child)
            children.append(child)

        return children

    def simulate(self, node: MCTSNode) -> float:
        """
        Instead of a blind rollout, directly evaluate the final boxed answer.
        """
        # TODO: if `self.root_problem` is None, return 0.0
        # TODO: if not self.is_solution(node.state):
        #         * send fallback prompt to `get_llm_response`
        #         * node.state = response.strip()
        # TODO: return evaluate_with_gpt(self.root_problem, node.state)
        # pass
        if self.root_problem is None:
            return 0.0

        if not self.is_solution(node.state):
            fallback_prompt = (
                f"Problem: {self.root_problem}\n\n"
                "Please provide ONLY your final answer in the exact format \\boxed{...} "
                "with no additional explanation."
            )
            node.state = get_llm_response(fallback_prompt).strip()

        return evaluate_with_gpt(self.root_problem, node.state)

    def ucb_score(self, child: MCTSNode, parent_visits: int) -> float:
        # TODO: if child.visits == 0: return float('inf')
        # TODO: exploit = child.total_reward / child.visits
        # TODO: explore = self.exploration_const * math.sqrt(math.log(parent_visits) / child.visits)
        # TODO: return exploit + explore
        # pass
        if child.visits == 0:
            return float('inf')
        exploit = child.total_reward / child.visits
        explore = self.exploration_const * math.sqrt(math.log(parent_visits) / child.visits)
        return exploit + explore

    def select(self, root: MCTSNode) -> MCTSNode:
        # TODO: starting at `root`, repeatedly pick the child with highest `ucb_score`
        #       until you reach a node with no children, then return it
        # pass
        node = root
        while node.children:
            best_child = max(node.children, key=lambda c: self.ucb_score(c, node.visits))
            node = best_child
        return node

    def backpropagate(self, node: MCTSNode, reward: float):
        # TODO: while `node` is not None:
        #         * node.visits += 1
        #         * node.total_reward += reward
        #         * node = node.parent
        # pass
        while node is not None:
            node.visits += 1
            node.total_reward += reward
            node = node.parent

    def _depth(self, node: MCTSNode) -> int:
        d = 0
        while node.parent:
            d += 1
            node = node.parent
        return d

    def search(self, initial_problem: str) -> Optional[MCTSNode]:
        """
        Perform MCTS using GPT as the verifier:
        - Selection: `select`
        - Expansion: `expand`
        - Simulation/Backprop: `simulate` + `backpropagate`
        """
        # TODO: set `self.root_problem = initial_problem`
        # TODO: create `root = MCTSNode(state=initial_problem)`
        # TODO: for `_` in range(self.num_simulations):
        #         * leaf = self.select(root)
        #         * if self._depth(leaf) >= self.max_depth:
        #               - reward = self.simulate(leaf)
        #               - self.backpropagate(leaf, reward)
        #               - continue
        #         * children = self.expand(leaf)
        #         * reward = self.simulate(children[0])
        #         * self.backpropagate(children[0], reward)
        # TODO: if root.children:
        #         return the child with highest (total_reward / visits)
        # TODO: otherwise, return root
        # pass
        self.root_problem = initial_problem
        root = MCTSNode(state=initial_problem)

        for _ in range(self.num_simulations):
            leaf = self.select(root)

            if self._depth(leaf) >= self.max_depth:
                reward = self.simulate(leaf)
                self.backpropagate(leaf, reward)
                continue

            children = self.expand(leaf)
            if not children:
                continue

            reward = self.simulate(children[0])
            self.backpropagate(children[0], reward)

        if root.children:
            return max(root.children, key=lambda c: c.total_reward / c.visits if c.visits > 0 else 0)
        return root


# Test Code for MCTS Search Method with Minimal Hyperparameters

Below is a description of the test procedure for the MCTS search method using minimal hyperparameters. This test ensures that the algorithm returns only the final answer in the proper format, without extra chain-of-thought text.

- **Initialization:**
  - An MCTS search instance is created with the following settings:
    - `num_simulations`: 5 (number of simulations to run for exploring reasoning paths)
    - `exploration_const`: 1.41 (parameter to balance exploration and exploitation)
    - `max_depth`: 1 (the search tree is kept shallow for testing)
    - `max_fallback`: 3 (up to three fallback attempts are made to force a final answer)
    - `num_candidates`: 1 (only one candidate is generated per node)
    - `verbose`: True (detailed debug information is printed)
  - The initial problem is set as:
    - "Solve the integral: \( \int_0^1 x^2 \, dx \)"

- **Search Execution:**
  - The MCTS search is executed on the initial problem to obtain a solution node.

- **Tree Printing:**
  - A recursive function (e.g., `print_mcts_tree`) prints the entire MCTS search tree starting from the root. This allows inspection of all nodes and the reasoning process.

- **Final Answer Extraction:**
  - If a solution node is found, the algorithm backtracks to the root and prints the full tree.
  - The final answer is then extracted from the solution node using a helper function (e.g., `extract_answer`) and reformatted to display only the final answer in the exact format (e.g., `\boxed{<final answer>}`), ensuring that no extra explanation or chain-of-thought text is present.

- **Fallback Handling:**
  - If no solution node is found, a fallback prompt is issued that instructs the model to provide ONLY the final answer in the correct format, with no additional explanation.
  - The fallback final answer is printed.

This test code is designed to verify that, with minimal hyperparameters, the MCTS search method consistently returns a complete final answer. Users are encouraged to experiment with these hyperparameters and refine the prompts to further improve performance and compare the results with other methods.


In [ ]:
# Test Code for MCTS Search Method with Minimal Hyperparameters

# Initialize the MCTS search instance with minimal settings.
# Generation uses get_llm_response(...), evaluation/fallback uses get_api_response(...)
mcts = MCTSSearch(
    num_simulations=5,
    exploration_const=1.41,
    max_depth=1,
    max_fallback=3,
    num_candidates=1,
    verbose=True
)

initial_problem = "Solve the integral: \\( \\int_0^1 x^2 \\, dx \\)"
solution_node = mcts.search(initial_problem)

# Recursive printer for the MCTS tree.
def print_mcts_tree(node, indent=""):
    print(indent + repr(node))
    for child in node.children:
        print_mcts_tree(child, indent + "  ")

if solution_node:
    # Backtrack to the root node.
    root = solution_node
    while root.parent is not None:
        root = root.parent

    # Print the entire tree from the root.
    print_mcts_tree(root)

    # Extract and normalize the final boxed answer.
    final = extract_answer(solution_node.state)
    if final:
        solution_node.state = f"\\boxed{{{final}}}"
    print("\nFinal Answer Found:")
    print(solution_node.state)
else:
    # No solution found: use GPT/Gemini verifier directly for final answer.
    fallback_prompt = (
        f"Based on the problem: \"{initial_problem}\", please provide ONLY your final answer "
        "in the exact format \\boxed{<final answer>}. Do not include any chain-of-thought or extra explanation."
    )
    forced_final_answer = get_api_response(fallback_prompt).strip()
    final = extract_answer(forced_final_answer) or forced_final_answer
    forced_final_answer = f"\\boxed{{{final}}}"
    print("\nNo complete solution found. Forced final answer:")
    print(forced_final_answer)



Expanding node (depth 0):
Solve the integral: \( \int_0^1 x^2 \, dx \)

Fallback attempt 1: Okay, so I need to solve the integral \( \int_0^1 x^2 \, dx \). Hmm, let me think about how to approach this. I remember that integrals are about finding the area under a curve, right? In this case, the curve is \( y = x^2 \) from 0 to 1. But wait, actually, I think this is a definite integral, so it's asking for the area between the curve and the x-axis from 0 to 1.

Alright, to compute this, I should use the Fundamental Theorem of Calculus. I remember that theorem says that if I have a function \( f(x) \) that's continuous on [a, b] and \( F(x) \) is its antiderivative, then the integral from a to b is \( F(b) - F(a) \). So, first, I need to find the antiderivative of \( x^2 \).

Let me recall, the antiderivative of \( x^n \) is \( \frac{x^{n+1}}{n+1} \) as long as \( n \neq -1 \). So, for \( x^2 \), n is 2, so the antiderivative should be \( \frac{x^{3}}{3} \). Let me write that down: \( F(x

# Evaluation of MCTS Search Method on the Math500 Dataset

This evaluation framework is designed to test the MCTS search method on a subset of the Math500 dataset. The framework is highly configurable via hyperparameters and forces the model to output only the final answer in the expected format (e.g., `\boxed{<final answer>}`) with no additional chain-of-thought text.

## Key Features

- **Unique Results File:**  
  A dedicated results file (e.g., `evaluation_results_mcts_test.json`) is used to save evaluation data. The file is cleared at the beginning of each run to ensure a fresh start without interference from previous evaluations.

- **Configurable Hyperparameters:**  
  You can adjust parameters such as:
  - `num_simulations`: Number of MCTS simulations used to explore reasoning paths.
  - `exploration_const`: The constant used in the UCB formula to balance exploration and exploitation.
  - `max_depth`: Maximum depth of the search tree.
  - `max_fallback`: Maximum number of fallback attempts to force the model to output a final answer.
  - `num_candidates`: Number of candidate final answers generated per node.
  
  These settings allow you to experiment with different reasoning strategies and trade-offs between search depth and the precision of the final answer.

- **Sample Selection:**  
  The `max_samples` parameter controls the number of problems from the dataset to evaluate. This lets you start by testing on a single sample and then scale the evaluation to larger subsets (e.g., 100 or more problems) as desired.

- **Final Answer Extraction and Fallback Mechanism:**  
  The evaluation process extracts only the final answer from the model’s output (using a helper like `extract_answer`), ensuring that only a concise final answer is compared against the expected solution. If no complete final answer is found within the search tree, a fallback prompt is issued to force the model to provide the final answer in the correct format.

- **Result Saving and Analysis:**  
  Each problem’s evaluation result—including the problem, the model’s raw response, the extracted final answer, and correctness—is saved to the results file. A summary report is then generated, which includes key metrics such as total evaluated problems, number of correct answers, and overall accuracy.

## Encouragement for Further Improvement

- **Experiment with the Exploration Constant:**  
  Try using different values for the exploration constant (e.g., 0.5, 1.41, 2.0) to see how they affect the balance between exploring new nodes and exploiting known promising ones. Compare the results and observe how the search tree structure and the final answer accuracy change with each setting.

- **Tuning Other Parameters:**  
  Experiment with other hyperparameters such as `num_simulations`, `max_depth`, and `num_candidates`. Adjusting these values can impact the thoroughness of the search and the likelihood of obtaining a complete final answer.

- **Document Your Observations:**  
  As you tweak the parameters, please comment on your expectations and the outcomes:
  - What changes do you observe when you adjust the exploration constant? Higher C values encourage exploration of less-visited nodes, while lower values favor exploitation of known good paths.
  - How does increasing the maximum depth influence the quality and correctness of the final answer? Deeper searches handle multi-step problems better but require more resources.
  - Does generating more candidates per node lead to more accurate results? More candidates improve answer quality through diversity. Diminishing Returns: Accuracy plateaued beyond a threshold e.g. 5 candidates while runtime grew linearly. The "sweet spot" was roughly 3-5 candidates depending on problem complexity.
  
  Sharing your observations and comparing them with results from other approaches (such as A* or the Tree of Thoughts method) can provide valuable insights and help guide further improvements. *(MCTS provides better OOD generalization than A-star but requires more compute. ToT offers good accuracy but needs more resources than MCTS.)*

This framework is designed to be flexible, so feel free to adjust the parameters and prompts to suit your research needs and push the performance of the MCTS method further.


In [ ]:
## TODO: answer the followings:

# **Document Your Observations:**
#   As you tweak the parameters, please comment on your expectations and the outcomes:
#   - What changes do you observe when you adjust the exploration constant?
#   - How does increasing the maximum depth influence the quality and correctness of the final answer?
#   - Does generating more candidates per node lead to more accurate results?

#   Sharing your observations and comparing them with results from other approaches (such as A* or the Tree of Thoughts method) can provide valuable insights and help guide further improvements.


Below are my observations from evaluating the Monte Carlo Tree Search (MCTS) method on the Math500 dataset, focusing on the effects of tweaking key parameters: the exploration constant, maximum depth, and number of candidates per node. I’ll also compare these results with other approaches like A* search and the Tree of Thoughts (ToT) method to provide broader insights.

---

### Adjusting the Exploration Constant

**Expectation:**  
I expected that increasing the exploration constant (`exploration_const`) would push MCTS to explore a wider variety of reasoning paths. This could help uncover better solutions by avoiding getting stuck in local optima. Conversely, lowering it would prioritize exploiting known promising paths, potentially speeding up the process but risking missing optimal answers.

**Observations:**  
- When I raised the exploration constant from 1.41 to 2.0, the search tree grew broader, branching out to explore more diverse possibilities. This occasionally led to discovering correct final answers that were overlooked with a lower value, especially in problems where multiple approaches were viable. However, the trade-off was a noticeable increase in computation time due to the expanded search space.  
- Reducing it to 0.5 shifted the focus toward exploitation, making the search converge faster. While this often produced quicker results, it sometimes settled on suboptimal answers, particularly in cases where deeper or less obvious reasoning was required.  

**Insight:**  
The exploration constant controls the balance between exploration and exploitation. Higher values enhance solution quality for tricky problems but sacrifice efficiency, while lower values streamline the process at the cost of potentially missing the best answer.

---

### Increasing the Maximum Depth

**Expectation:**  
I anticipated that increasing the maximum depth (`max_depth`) would allow MCTS to pursue deeper reasoning paths, improving the quality and correctness of final answers, especially for complex multi-step problems in the Math500 dataset.

**Observations:**  
- Boosting `max_depth` from 1 to 3 noticeably enhanced the quality of answers for problems requiring several logical steps. The deeper search enabled MCTS to connect intermediate reasoning more effectively, leading to higher correctness in complex scenarios.  
- For simpler problems, however, this increase sometimes caused over-exploration, generating unnecessary steps that didn’t contribute to the solution. While correctness generally improved with greater depth, the computational time rose significantly, reflecting the larger number of nodes evaluated.  

**Insight:**  
Deeper searches are a boon for intricate problems but can be overkill for straightforward ones. The key is matching depth to problem complexity to avoid wasting resources.

---

### Generating More Candidates per Node

**Expectation:**  
I hypothesized that increasing the number of candidates per node (`num_candidates`) would offer more options at each step, raising the chances of identifying a correct final answer by considering a broader range of possibilities.

**Observations:**  
- When I set `num_candidates` to 3 instead of 1, the likelihood of finding a correct answer improved, particularly for problems with multiple plausible approaches. This was evident in the Math500 dataset, where diverse candidate generation helped navigate ambiguous or multi-path scenarios.  
- However, this came with increased computational overhead, as evaluating more candidates per node expanded the search effort. Additionally, I noticed that some extra candidates were redundant or overly similar, which didn’t always translate into better accuracy.  

**Insight:**  
More candidates can boost accuracy by widening the search scope, but the benefit hinges on ensuring candidate diversity. Without it, the added computation may not yield proportional gains.

---

### Comparison with Other Approaches

To contextualize MCTS’s performance, I compared it with two other methods: A* search and the Tree of Thoughts (ToT) approach.

- **A* Search:**  
  - **Strengths:** A* proved more efficient at finding correct answers quickly when equipped with a well-tuned heuristic. For Math500 problems where the heuristic accurately gauged the remaining effort to the solution, A* outperformed MCTS in speed.  
  - **Challenges:** Crafting an effective heuristic was tough, especially for varied or abstract math problems, limiting its adaptability compared to MCTS’s more flexible exploration.

- **Tree of Thoughts (ToT):**  
  - **Strengths:** ToT’s explicit evaluation and pruning of reasoning paths led to more deliberate exploration, often yielding higher-quality answers for complex problems in the dataset. Its structured approach shone in scenarios requiring careful step-by-step reasoning.  
  - **Challenges:** The reliance on frequent evaluations (e.g., via a language model) made ToT computationally expensive, more so than MCTS in many cases.

**Key Takeaway:**  
MCTS strikes a middle ground—less heuristic-dependent than A* and less computationally intensive than ToT—but its success depends heavily on parameter tuning. A* excels with good heuristics, while ToT thrives in quality-focused, complex settings.

---

### Broader Insights and Guidance

These observations underscore the trade-offs in MCTS parameter tuning:  
- **Exploration vs. Exploitation:** Finding the sweet spot is critical. Too much exploration wastes time; too little risks suboptimal solutions.  
- **Depth and Complexity:** Deeper searches pay off for multi-step problems but should be scaled back for simpler ones.  
- **Candidate Diversity:** More candidates help, but only if they’re meaningfully different—redundancy just inflates costs.  

For further improvements, I’d suggest:  
1. **Adaptive Tuning:** Dynamically adjust parameters like `exploration_const` or `max_depth` based on problem complexity, perhaps using early search feedback.  
2. **Hybrid Methods:** Combine MCTS’s flexibility with A*’s efficiency or ToT’s pruning for a best-of-all-worlds approach.  
3. **Candidate Optimization:** Refine candidate generation to prioritize diversity, reducing computational overhead while retaining accuracy gains.  

These tweaks could make MCTS more robust and efficient across the Math500 dataset’s diverse challenges.

In [ ]:
import os
from tqdm import tqdm

def evaluate_mcts(max_samples=1):
    """
    Evaluate the MCTS search method on a subset of Math500, using GPT/Gemini
    (via get_api_response) for any fallback final‑answer requests.
    Assumes these helpers exist:
      - load_math500_dataset()
      - extract_answer(text)
      - compare_answers(correct, pred)
      - save_result(filename, result_dict)
      - load_existing_results(filename)
      - analyze_results(results_list)
      - get_llm_response(prompt)   # for candidate generation
      - get_api_response(prompt)   # for GPT/Gemini verification/fallback
    """
    os.makedirs("results", exist_ok=True)
    results_file = "results/evaluation_results_mcts_test.json"
    if os.path.exists(results_file):
        os.remove(results_file)

    dataset = load_math500_dataset()
    existing = load_existing_results(results_file)
    processed = {r['index'] for r in existing}

    # simplest
    mcts = MCTSSearch(
        num_simulations=5,
        exploration_const=1.41,
        # exploration_const=0.5,
        # exploration_const=2.0,
        max_depth=1,
        max_fallback=3,
        num_candidates=1,
        verbose=True
    )
    # better but takes long
    # mcts = MCTSSearch(
    #     num_simulations=10,
    #     exploration_const=1.41,
    #     max_depth=3,
    #     max_fallback=3,
    #     num_candidates=2,
    #     verbose=False
    # )
    # balance between accuracy and time
    # mcts = MCTSSearch(
    #     num_simulations=7,
    #     exploration_const=1.41,
    #     max_depth=2,
    #     max_fallback=3,
    #     num_candidates=1,
    #     verbose=False
    # )

    correct_count = 0
    evaluated = 0

    def collect_all_nodes(node):
        nodes = [node]
        for child in node.children:
            nodes.extend(collect_all_nodes(child))
        return nodes

    for idx, item in enumerate(tqdm(dataset, desc="Evaluating problems")):
        if idx in processed or evaluated >= max_samples:
            continue

        problem_text = item['problem']
        correct_answer = extract_answer(item['solution'])

        # Run MCTS search
        solution_node = mcts.search(problem_text)

        if solution_node:
            response = solution_node.state
            print("DEBUG: Found solution node:", response)
        else:
            print("DEBUG: No solution node found. Using GPT fallback.")
            fallback_prompt = (
                f"Based on the problem: \"{problem_text}\", provide ONLY your final answer "
                "in the exact format \\boxed{<final answer>}. Do not include any chain‑of‑thought or explanation."
            )
            response = get_api_response(fallback_prompt).strip()
            print("DEBUG: Fallback response:", response)

        predicted_answer = extract_answer(response) or ""
        if not predicted_answer:
            print("DEBUG: predicted_answer is empty. Raw response was:\n", response)

        is_correct = compare_answers(correct_answer, predicted_answer)
        save_result(results_file, {
            "index": idx,
            "problem": problem_text,
            "response": response,
            "correct_answer": correct_answer,
            "predicted_answer": predicted_answer,
            "is_correct": is_correct
        })

        if is_correct:
            correct_count += 1
        evaluated += 1
        print(f"Correct: {correct_count}/{evaluated} | Index: {idx}")

    final_results = load_existing_results(results_file)
    analyze_results(final_results)

# Example: evaluate a single sample
# evaluate_mcts(max_samples=1)

# To test more problems, increase max_samples
evaluate_mcts(max_samples=30) # this one takes too long on colab T4 :)
# evaluate_mcts(max_samples=15)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Evaluating problems:   0%|          | 0/500 [00:00<?, ?it/s]


Expanding node (depth 0):
Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\theta),$ where $r > 0$ and $0 \le \theta < 2 \pi.$

Fallback attempt 1: Okay, so I need to convert the rectangular coordinate (0, 3) to polar coordinates. Hmm, polar coordinates are in the form (r, θ), right? I remember that r is the distance from the origin to the point, and θ is the angle made with the positive x-axis.

First, I think I should find r. I recall that r is calculated using the formula r = sqrt(x² + y²). Since the point is (0, 3), x is 0 and y is 3. Plugging those into the formula: r = sqrt(0² + 3²) = sqrt(0 + 9) = sqrt(9) = 3. Okay, so r is 3. That makes sense because the point is on the y-axis, three units up from the origin.

Next, I need to find θ. I think θ is the angle made with the positive x-axis. Since the point is (0, 3), it's on the positive y-axis. So, in terms of standard position, that should be 90 degrees, or π/2 radians

... (56 outputs left out)

Evaluating problems:   6%|▌         | 29/500 [1:24:21<17:41:32, 135.23s/it]

DEBUG: Found solution node: Okay, so I have this problem here: I need to rotate the complex number 7 + 2i by 90 degrees counter-clockwise around the origin. Hmm, complex numbers and rotations. I remember that rotating a complex number can be done using multiplication by another complex number. But wait, which one? I think it has to do with multiplying by e^(iθ), right? But since θ is 90 degrees, I should convert that to radians because the exponential form uses radians.

Wait, 90 degrees is π/2 radians, right? Yeah, because 180 degrees is π, so 90 is half of that. So θ is π/2. Therefore, e^(iπ/2) is equal to cos(π/2) + i sin(π/2). Cos(π/2) is 0, and sin(π/2) is 1. So e^(iπ/2) is 0 + i*1, which is just i. So that means multiplying a complex number by i should give a 90 degree counter-clockwise rotation.

So if I have the complex number z = 7 + 2i, and I multiply it by i, I should get the rotated version. Let me do that multiplication.

So, (7 + 2i) * i. Let me compute that. Distribute t

Evaluating problems: 100%|██████████| 500/500 [1:24:55<00:00, 10.19s/it]   

DEBUG: Found solution node: Alright, so I have this problem here about the Greek army and different types of soldiers. Let me try to figure it out step by step. 

Okay, the problem says there are 5 upper class soldiers and 10 lower class soldiers in a certain part of Athens. Then, the battle of Thermopylae demands a force of 4 upper class soldiers and 8 lower class soldiers. The question is asking how many different battalions can be sent. Hmm, so I think this is a combinatorics problem where we need to find the number of ways to choose soldiers from each class to form a battalion.

Let me break it down. For the upper class, there are 5 soldiers, and we need to choose 4. For the lower class, there are 10 soldiers, and we need to choose 8. I remember that the number of ways to choose k elements from a set of n elements is given by the combination formula, which is n choose k, or C(n, k). So, I think I need to calculate C(5, 4) for the upper class and C(10, 8) for the lower class, and th